# Introduction
The following program was authored by Max Tejera in 2023-2024. The program was written to conduct analysis on the impacts of the Colorado Equal Pay for Equal Work Act (EPEWA) on the gender wage gap in Colorado. The original analysis was for Max's master of economics capstone research project at the University of Colorado Denver during the fall term of 2023. The code was originally authored in Stata, but has been translated to Python here with some additional functionality.

**Update (October 2026):** The analysis was refreshed in October 2026 to extend the sample from May 2024 through September 2026. The hardcoded file paths, sample names, minimum wage tables and industry crosswalk were replaced with code that reads them from current sources. The original July 2024 window can still be reproduced by setting `SAMPLE_END = (2024, 5)` in the configuration cell. Results for both windows are saved side by side for comparison.

The refresh also responds to a methodological review: Section 6 tests whether the result survives alternative earnings cuts and a placebo test, California and Washington are measured from their own January 2023 start date, and gaps are reported as exact cents on the dollar, exp(β) rather than 1 + β.

# Section 0: Setup
The first block imports the libraries used in the project. The second is a configuration cell. It sets:

- the data folder and the location of the IPUMS API key, read from the environment variables `EPEWA_DATA_DIR` and `IPUMS_API_KEY_FILE` so no personal paths or keys are stored in the notebook;
- the sample window (`SAMPLE_START`, `SAMPLE_END`);
- `SUBMIT_NEW_EXTRACT`, which should be `True` only when a new IPUMS extract is needed, so that re-running the notebook never files duplicate requests;
- a connection setting for running behind RAND's network (see the README)
- the helper functions used for regressions and charts.

In [1]:
import bisect
import gzip
import os
import re
import shutil
import ssl
import warnings
from datetime import datetime
from pathlib import Path
import certifi
import numpy as np
import pandas as pd
import plotly.colors as pc
import plotly.graph_objects as go
import requests
import statsmodels.formula.api as smf
from bs4 import BeautifulSoup
from ipumspy import IpumsApiClient, MicrodataExtract, readers
from requests.adapters import HTTPAdapter

In [2]:
# ---- Project configuration -------------------------------------------------
# Paths come from environment variables so the notebook runs on any machine.
DATA_DIR = Path(os.environ.get("EPEWA_DATA_DIR", Path.cwd() / "data"))
IPUMS_API_KEY_FILE = os.environ.get("IPUMS_API_KEY_FILE")   # text file holding the key (or set IPUMS_API_KEY)

# Study window for the IPUMS CPS monthly samples.
# (2024, 5) reproduces the July 2024 thesis run; set to the latest month to refresh.
SAMPLE_START = (2010, 1)
SAMPLE_END   = (2026, 9)

# Set True only when you need a fresh IPUMS extract (new sample window or new data release).
# False reuses the newest extract already in DATA_DIR, so "Run All" never files duplicate requests.
SUBMIT_NEW_EXTRACT = False

DATA_DIR.mkdir(parents=True, exist_ok=True)
download_dir = DATA_DIR

# Earnings cut for the main analysis: the CPS topcode for the outgoing rotation groups through March 2024
# ($150,000 / 52 = $2,884.61 a week). Section 6 tests alternative cuts.
TOPCODE_WEEKLY = 2884.61

In [3]:
# ---- TLS workaround for RAND's Cisco Secure Access proxy ---------------------------------------------
# Behind RAND's network, HTTPS is re-signed by Cisco's root. Python 3.13+ also demands an Authority Key
# Identifier on every certificate, which that chain lacks. This keeps full certificate verification
# (against the certifi bundle that has RAND_PKI_Root.pem appended) and only drops that one extra strict flag.


class RelaxedStrictAdapter(HTTPAdapter):
    def init_poolmanager(self, *args, **kwargs):
        ctx = ssl.create_default_context(cafile=certifi.where())
        ctx.verify_flags &= ~ssl.VERIFY_X509_STRICT
        kwargs['ssl_context'] = ctx
        return super().init_poolmanager(*args, **kwargs)

_session = requests.Session()
_session.mount('https://', RelaxedStrictAdapter())
requests.get = _session.get          # every requests.get in the notebook now uses this session

### Helper functions
Reusable functions used by the later sections: `fit_did` and `run_by_group` fit the difference-in-differences regression for one group or many (the optional `post_col` argument lets a group use its own post-law date), `term_table` prints selected coefficients, and `plot_group_gaps` and `plot_state_bars` draw the charts. `run_by_group` also converts the coefficients to exact "cents on the dollar" ratios: `pre_epewa_wage_gap` = exp(female coefficient), `post_epewa_wage_gap` = exp(female + interaction), and `ratio_change` = their difference.

In [4]:
# ---- Regression helpers ---------------------------------------------------
KEY_FEMALE = 'C(female)[T.1]'
KEY_INTERACTION = 'C(female)[T.1]:C(epewa)[T.1]'

FORMULA_NO_IND = 'log_earnyear ~ C(female) + C(epewa) + C(female):C(epewa) + C(age) + C(race_cat) + C(educ)'
FORMULA_IND = FORMULA_NO_IND + ' + C(ind_cat20_label)'

def fit_did(data, formula, post_col=None):
    """OLS with HC3 robust standard errors. Returns the model and the key coefficients.
    post_col: optional name of a 0/1 column to use as the post-law period instead of `epewa`
    (used to give California and Washington their own January 2023 start)."""
    if post_col is not None:
        data = data.assign(epewa=data[post_col])
    with warnings.catch_warnings():
        # Subsets (one state, one industry) can be missing some category levels; statsmodels then reports a
        # rank-deficient design matrix. The coefficients of interest are still estimated, so the warning is silenced.
        warnings.filterwarnings('ignore', message='The design matrix is rank-deficient')
        model = smf.ols(formula, data=data).fit(cov_type='HC3')
    return model, {
        'female_coef': model.params[KEY_FEMALE],
        'female_pval': model.pvalues[KEY_FEMALE],
        'interaction_coef': model.params[KEY_INTERACTION],
        'interaction_pval': model.pvalues[KEY_INTERACTION],
        'interaction_se': model.bse[KEY_INTERACTION],
    }

def run_by_group(df, group_col, formula, min_obs=2, include_national=False, add_gaps=True, post_col=None):
    """Fit the model separately for each group (optionally plus one row for the whole sample).
    Returns {group: model} and a results table. With add_gaps=True (log earnings), it also adds the
    'cents on the dollar' pre- and post-law gaps. post_col: see fit_did."""
    models, rows = {}, {}
    for group, subset in df.groupby(group_col, observed=True):
        if len(subset) >= min_obs:
            models[group], rows[group] = fit_did(subset, formula, post_col)
    if include_national:
        models['United States'], rows['United States'] = fit_did(df, formula, post_col)
    out = pd.DataFrame.from_dict(rows, orient='index')
    if add_gaps:
        # Exact "cents on the dollar": women's earnings as a share of men's is exp(coefficient), not 1 + coefficient
        out['pre_epewa_wage_gap'] = np.exp(out['female_coef'])
        out['post_epewa_wage_gap'] = np.exp(out['female_coef'] + out['interaction_coef'])
        out['ratio_change'] = out['post_epewa_wage_gap'] - out['pre_epewa_wage_gap']
    return models, out

def term_table(model, patterns):
    """Coefficient, standard error and p-value for the model terms whose names contain any of the given strings."""
    names = [t for t in model.params.index if any(p in t for p in patterns)]
    return pd.DataFrame({'coef': model.params[names], 'std_err': model.bse[names], 'p_value': model.pvalues[names]}).round(4)

In [5]:
# ---- Chart helpers (dark theme), used by Sections 4 and 5 -------------------
# Period labels come from the config cell, so they update when SAMPLE_END changes.
PRE_PERIOD = "Jan 2010 - Dec 2020"
POST_PERIOD = f"Jan 2021 - {pd.Timestamp(SAMPLE_END[0], SAMPLE_END[1], 1):%b %Y}"

# Every chart is also saved as a standalone HTML file here (GitHub does not display Plotly charts inside notebooks).
CHART_DIR = DATA_DIR / 'results' / f"{SAMPLE_START[0]}-{SAMPLE_START[1]:02d}_{SAMPLE_END[0]}-{SAMPLE_END[1]:02d}" / 'charts'

# Highlighted jurisdictions get a fixed accent color; every other state is colored by its value.
HIGHLIGHT_COLORS = {'Colorado': '#34d399', 'California': '#fbbf24', 'Washington': '#60a5fa', 'United States': '#f472b6'}
GROUP_COLORS = {**HIGHLIGHT_COLORS, 'Rest of US': '#94a3b8'}   # Section 4 comparison groups
BAR_SCALE = ['#334155', '#3b6ea5', '#22d3ee']                  # slate -> blue -> cyan, low to high value
BG_TOP, BG_BOTTOM = (8, 12, 22), (30, 41, 59)                   # background gradient, top to bottom (RGB)
FONT = "Bahnschrift, 'Segoe UI Variable Display', 'Segoe UI', Helvetica, Arial, sans-serif"   # Bahnschrift ships with Windows 10/11
ROW_HEIGHT = 22                                                 # pixels per state, so every state fits


def _save_chart(fig, name):
    """Write the figure to CHART_DIR/<name>.html (small file; loads Plotly's JavaScript from the web)."""
    CHART_DIR.mkdir(parents=True, exist_ok=True)
    fig.write_html(CHART_DIR / f"{name}.html", include_plotlyjs='cdn', config={'displayModeBar': False})


def _gradient_bands(top, bottom, n=48):
    """Fake a vertical gradient (Plotly has no native gradient background). Each rectangle runs from the
    bottom edge up to its own top edge and is painted over the one before it, so there are no visible seams."""
    bands = []
    for i in range(n):
        t = i / (n - 1)
        rgb = tuple(round(a + (b - a) * t) for a, b in zip(top, bottom))
        bands.append(dict(type='rect', xref='paper', yref='paper', x0=0, x1=1,
                          y0=0, y1=1 - i / n, fillcolor=f"rgb{rgb}", line_width=0, layer='below'))
    return bands


def _hex_to_rgba(hex_color, alpha):
    h = hex_color.lstrip('#')
    return f"rgba({int(h[0:2], 16)},{int(h[2:4], 16)},{int(h[4:6], 16)},{alpha})"


def _apply_dark_theme(fig, headline, controls, period, height, footnote):
    """Shared look: title, fonts, gradient background, footnote."""
    fig.update_layout(
        title=dict(text=f"<b>{headline}</b><br><span style='font-size:14px;color:#94a3b8'>Controls: {controls}"
                        f"<br>{period}</span>", x=0.5, xanchor='center', y=1, yanchor='top', pad=dict(t=40),
                   font=dict(size=24, color='#f8fafc')),
        font=dict(family=FONT, color='#cbd5e1'),
        paper_bgcolor='#0b1220', plot_bgcolor='rgba(0,0,0,0)', shapes=_gradient_bands(BG_TOP, BG_BOTTOM),
        height=height, margin=dict(l=20, r=60, t=170, b=120))
    fig.add_annotation(text=footnote, x=0.5, y=0, yanchor='top', yshift=-70, xref='paper', yref='paper',
                       showarrow=False, font=dict(size=12, color='#94a3b8'))


def plot_state_bars(results, value_col, pval_col, headline, controls, period, x_range, save_as=None):
    """Section 5: horizontal bar chart of one value per state, sorted ascending. A state gets an asterisk
    when its p-value (pval_col) is above 0.10. If save_as is given, the chart is also saved as HTML.
    *Not statistically significant at the 10% level (p > 0.10)<br>California and Washington: post period starts Jan 2023"""
    d = results.sort_values(value_col, ascending=True)
    spread = d[value_col].max() - d[value_col].min()
    norm = ((d[value_col] - d[value_col].min()) / spread) if spread else d[value_col] * 0 + 0.5
    base = pc.sample_colorscale(BAR_SCALE, list(norm))
    colors = [HIGHLIGHT_COLORS.get(state, c) for state, c in zip(d.index, base)]
    outline = ['#f8fafc' if state in HIGHLIGHT_COLORS else 'rgba(0,0,0,0)' for state in d.index]
    y_labels = [f"<b>{state}{'*' if pval > 0.10 else ''}</b>" for state, pval in zip(d.index, d[pval_col])]
    text = [f"<b>{v * 100:,.1f}%</b>" for v in d[value_col]]

    fig = go.Figure(go.Bar(
        x=d[value_col], y=y_labels, orientation='h', text=text, textposition='outside',
        textfont=dict(color='#f1f5f9', size=12),
        marker=dict(color=colors, line=dict(color=outline, width=1.5)),
        hovertemplate='%{y}: %{text}<extra></extra>', cliponaxis=False))
    _apply_dark_theme(fig, headline, controls, period, 260 + ROW_HEIGHT * len(d),
                      "*Not statistically significant at the 10% level (p > 0.10)")
    x_dtick = 0.02 if (x_range[1] - x_range[0]) <= 0.2 else 0.05   # fixed tick spacing so labels never collide
    fig.update_layout(
        bargap=0.25,
        xaxis=dict(range=x_range, tickformat='.0%', dtick=x_dtick, tickangle=0, showgrid=True,
                   gridcolor='rgba(148,163,184,0.15)', zeroline=True, zerolinecolor='rgba(148,163,184,0.45)',
                   showline=False, tickfont=dict(size=13, color='#cbd5e1', weight='bold')),
        yaxis=dict(type='category', dtick=1, showgrid=False, showline=False,
                   tickfont=dict(size=12, color='#f1f5f9'), automargin=True))
    if save_as:
        _save_chart(fig, save_as)
    fig.show()
    return fig


def plot_group_gaps(results, headline, controls, save_as=None):
    """Section 4: pre- and post-law wage gap for each comparison group, sorted by the size of the change.
    Faded bars: before the law  |  Solid bars: after the law  |  Boxed value: change in cents on the dollar.
    If save_as is given, the chart is also saved as HTML."""
    assert len(results) < 10, "results_df holds the state table. Run the Section 4 regression cell directly above this chart first."
    d = results.copy()
    d['pre_epewa_wage_gap'] = np.exp(d['female_coef'])                       # exact cents on the dollar
    d['post_epewa_wage_gap'] = np.exp(d['female_coef'] + d['interaction_coef'])
    d['ratio_change'] = d['post_epewa_wage_gap'] - d['pre_epewa_wage_gap']
    d = d.sort_values('ratio_change', ascending=False)
    groups = list(d.index)
    accent = [GROUP_COLORS.get(g, '#94a3b8') for g in groups]

    def bars(name, col, colors):
        return go.Bar(name=name, x=[f"<b>{g}</b>" for g in groups], y=d[col], marker=dict(color=colors, line_width=0),
                      text=[f"<b>{v * 100:,.1f}%</b>" for v in d[col]], textposition='outside',
                      textfont=dict(color='#f1f5f9', size=13), cliponaxis=False,
                      hovertemplate=f'%{{x}} ({name}): %{{text}}<extra></extra>')

    fig = go.Figure([bars('Before the law', 'pre_epewa_wage_gap', [_hex_to_rgba(c, 0.45) for c in accent]),
                     bars('After the law', 'post_epewa_wage_gap', accent)])
    _apply_dark_theme(fig, headline, controls, f"{PRE_PERIOD} vs. {POST_PERIOD}", 720,
                    "Faded bars: before the law  |  Solid bars: after the law  |  Boxed value: change in cents on the dollar"
                    "<br>California and Washington: post period starts Jan 2023")
    for g, c, change in zip(groups, accent, d['ratio_change']):
        fig.add_annotation(x=f"<b>{g}</b>", y=0.975, xref='x', yref='y', text=f"<b>{change * 100:+.1f}¢</b>",
                           showarrow=False, font=dict(size=14, color=c), bgcolor='rgba(11,18,32,0.85)',
                           bordercolor=c, borderwidth=1.5, borderpad=5)
    fig.update_layout(
        barmode='group', bargap=0.35, bargroupgap=0.08, showlegend=False,
        xaxis=dict(showgrid=False, tickfont=dict(size=14, color='#f1f5f9')),
        yaxis=dict(range=[0.7, 1], tickformat='.0%', dtick=0.05, showgrid=True, gridcolor='rgba(148,163,184,0.15)',
                   showline=False, tickfont=dict(size=13, color='#cbd5e1', weight='bold')))
    if save_as:
        _save_chart(fig, save_as)
    fig.show()
    return fig

# Section 1: Load Current Population Survey Data via API
This section uses the IPUMSPY library from IPUMS and an API key saved outside the project folder to load the selected sample from the Current Population Survey (CPS) into a Pandas dataframe.

The selected sample is every monthly sample from January 2010 through September 2026 (200 samples). IPUMS has no October 2025 sample because those data were not collected during the federal government shutdown. Later on the sample will be divided into two time periods by the EPEWA variable (Equal Pay for Equal Work Act). EPEWA was passed during the Colorado 2019 legislative session and came into effect January 2021. To create a before and after comparison, the pre-law period was defined as January 2010 to December 2020. This period was chosen because the U.S. economy was mostly stable during this time, with some major abberations in the latter 10 months of 2020. This pre-law period is intended to capture a pre-trend in the gender wage gap prior to EPEWA coming into effect. The larger sample of 2010-2019 period likely offsets any major disruptions from the pandemic.

The CPS variables selected are elaborated on in the later sections. The number of variables selected has been minimized to optimize loading time. The first load for a given sample window takes approximately 30 minutes. I believe this is due to how IPUMSPY maps the data while loading. The eligible sample is then cached to disk under a name that includes the sample window, so later runs load in seconds. It has no upper earnings cut: that is applied after loading (Section 2), so Section 6 can compare alternatives.

The sample names are pulled from the IPUMS API and filtered to the study window set in the configuration cell. A check lists any months missing from the window.

IPUMS requires all researchers to cite them when using their data. They are cited below and within the code.

##### Citation
> Sarah Flood, Miriam King, Renae Rodgers, Steven Ruggles, J. Robert Warren, Daniel Backman, Etienne Breton, Grace Cooper, Julia A. Rivera Drew, Stephanie Richards, David Van Riper, and Kari C.W. Williams. IPUMS CPS: Version 13.0 [dataset]. Minneapolis, MN: IPUMS, 2025. https://doi.org/10.18128/D030.V13.0

In [6]:
# Open IPUMS API. The key is never stored in the repo.
api_key = os.environ.get("IPUMS_API_KEY")
if not api_key:
    if not IPUMS_API_KEY_FILE:
        raise RuntimeError("Set the IPUMS_API_KEY or IPUMS_API_KEY_FILE environment variable (see README).")
    api_key = Path(IPUMS_API_KEY_FILE).read_text().strip()
ipums = IpumsApiClient(api_key)
ipums.session.mount('https://', RelaxedStrictAdapter())   # the TLS workaround from the setup section, applied to the IPUMS client

In [7]:
# Get the CPS monthly sample IDs directly from the IPUMS API
# (replaces the BeautifulSoup scrape of the sample_ids web page).
# The study window (SAMPLE_START / SAMPLE_END) is set in the config cell.
all_samples = ipums.get_all_sample_info("cps")   # {sample_id: description}

pattern = re.compile(r"^cps(\d{4})_(\d{2})[a-z]$")   # e.g. cps2010_01s

filtered_sample_dict = {}
for sample_id, description in all_samples.items():
    m = pattern.match(sample_id)
    if not m:
        continue
    if "ASEC" in description:   # skip the annual supplement samples
        continue
    ym = (int(m.group(1)), int(m.group(2)))
    if SAMPLE_START <= ym <= SAMPLE_END:
        filtered_sample_dict[sample_id] = description

CpsSamples = sorted(filtered_sample_dict, key=lambda s: (s[3:7], s[8:10]))
assert CpsSamples, "No samples found. Check the filters or the IPUMS API connection."
print(f"{len(CpsSamples)} samples: {CpsSamples[0][8:10]}/{CpsSamples[0][3:7]} to {CpsSamples[-1][8:10]}/{CpsSamples[-1][3:7]}")

# Flag any missing months in the window so gaps are not silent
expected, (y, mth) = [], SAMPLE_START
while (y, mth) <= SAMPLE_END:
    expected.append((y, mth)); mth += 1
    if mth > 12: y, mth = y + 1, 1
found = {(int(s[3:7]), int(s[8:10])) for s in CpsSamples}
missing = [f"{y}-{m:02d}" for y, m in expected if (y, m) not in found]
print("Missing months:", missing if missing else "none")

200 samples: 01/2010 to 09/2026
Missing months: ['2025-10']


In [8]:
CpsVariables = [
    "YEAR", 
    "MONTH", 
    "MISH",
    "STATEFIP", 
    "METFIPS", 
    "EARNWEEK2", 
    "AGE", 
    "SEX", 
    "RACE", 
    "HISPAN", 
    "IND", 
    "CLASSWKR",
    "WKSTAT", 
    "EDUC",
]

extract = MicrodataExtract(collection='cps', samples=CpsSamples, variables=CpsVariables)

print('Sarah Flood, Miriam King, Renae Rodgers, Steven Ruggles, J. Robert Warren, Daniel Backman, Annie Chen, Grace Cooper, Stephanie Richards, Megan Schouweiler, and Michael Westberry. IPUMS CPS: Version 11.0 [dataset]. Minneapolis, MN: IPUMS, 2023. https://doi.org/10.18128/D030.V11.0')

Sarah Flood, Miriam King, Renae Rodgers, Steven Ruggles, J. Robert Warren, Daniel Backman, Annie Chen, Grace Cooper, Stephanie Richards, Megan Schouweiler, and Michael Westberry. IPUMS CPS: Version 11.0 [dataset]. Minneapolis, MN: IPUMS, 2023. https://doi.org/10.18128/D030.V11.0


In [9]:
if SUBMIT_NEW_EXTRACT:
    extract_id = ipums.submit_extract(extract)
    print("Submitted extract:", extract_id)
    ipums.wait_for_extract(extract)          # can take several minutes
    print("Status:", ipums.extract_status(extract))
    ipums.download_extract(extract_id, download_dir=DATA_DIR)
    print("Extract downloaded to the data folder set in the config cell.")
else:
    print("SUBMIT_NEW_EXTRACT is False: reusing the newest extract already in DATA_DIR.")

SUBMIT_NEW_EXTRACT is False: reusing the newest extract already in DATA_DIR.


In [10]:
# Reference the most recently downloaded files
xml_files = sorted(download_dir.glob("*.xml"), key=lambda f: f.stat().st_mtime)
dat_gz_files = sorted(download_dir.glob("*.dat.gz"), key=lambda f: f.stat().st_mtime)
ddi_file = xml_files[-1]
dat_gz_file = dat_gz_files[-1]

# Read the DDI file, which maps the raw data
with warnings.catch_warnings():
    warnings.simplefilter('ignore')          # IPUMS's citation warning prints a local file path; the citation is printed below
    ddi = readers.read_ipums_ddi(ddi_file)
print(ddi.ipums_citation)

# Get the correct .dat file name from the DDI
dat_file = download_dir / ddi.file_description.filename

if not dat_file.name.endswith('.dat'):
    dat_file = dat_file.with_suffix('.dat')

# Check if the .dat file already exists
if not dat_file.exists():
    try:
        with gzip.open(dat_gz_file, 'rb') as f_in:
            with open(dat_file, 'wb') as f_out:
                shutil.copyfileobj(f_in, f_out)
            print(f"Decompressed data file saved to: {dat_file.name}")
    except EOFError as e:
        print(f"Error decompressing the file: {e}")
        print("The file might be corrupted or incomplete. Please try re-downloading the extract.")
        
# The next step uses the DDI file and the .dat data file to read the IPUMS data into Pandas for later manipulation.

Publications and research reports based on the IPUMS CPS database must cite it appropriately. The citation should include the following:

Sarah Flood, Miriam King, Renae Rodgers, Steven Ruggles, J. Robert Warren, Daniel Backman, Etienne Breton, Grace Cooper, Julia A. Rivera Drew, Stephanie Richards, David Van Riper, and Kari C.W. Williams. IPUMS CPS: Version 13.0 [dataset]. Minneapolis, MN: IPUMS, 2025. https://doi.org/10.18128/D030.V13.0

The licensing agreement for use of IPUMS CPS data requires that users supply us with the title and full citation for any publications, research reports, or educational materials making use of the data or documentation. Please add your citation to the IPUMS bibliography: http://bibliography.ipums.org/


The dataset is large and must be organized with the DDI file.
The next step breaks the database into small chunks and applies filters to the data to make this task faster.
The first load is slow (about 30 minutes), so the filtered result is cached to disk. To speed this process up, the dataset is pre-filtered as it is loaded. The original dataset has several million observations. The dataset ultimately used for analysis has approximately 600,000 observations (599,085 after the upper earnings cut in item 4, before the minimum wage exclusion in Section 2). Given so many are dropped, below is an explanation for the exclusions:

1. The CPS interviews respondents for four months, takes an eight month break, then interviews them again for four months. The CPS does this in rotation. Therefore, only one eigth of the data in the CPS represents a unique respondent. MISH==8 selects the last month of interviews for a given respondent. This is the last month in which the respondent is asked about their weekly earnings, the primary variable of interest.
2. To allow for a more direct comparison of male and female workers, only full-time workers (35+ hours / week) were considered. These are coded as WKSTAT==11.
3. Similarly, to allow for a more direct comparison of workers only 'Prime Working Age' workers are considered. This is to exclude workers that may have high variation due to school schedules (less than age 25) or may be experiencing wage penalties for approaching retirement (greater than 54).
4. The upper cut is applied after loading. EARNWEEK2 is the endogenous variable of interest. It is what the respondent reported earning from employment before taxes during the week of the 12th in the prior month. Starting in 2023, the CPS began rounding these values to preserve privacy of higher earning respondents. Any respondents making above 150,000 per year were rounded or topcoded to preserve privacy. This removes any variation above this income level and warrants excluding those observations in which EARNWEEK2 is the equivalent of 150,000 per year or more. This likely means all measures of the gender wage gap considered in the analysis are lower than in the population, as men tend to be more represented in higher income and leadership roles. The topcode threshold ($2,884.61 per week) is held fixed over the whole sample and is not adjusted for inflation.
5. Respondents reporting that they make less than the federal minimum wage equivalent (40 hours x 7.25 per hour = 290) were excluded to ensure respondents' answers are consistent with working full time with the more conventional 40 hour work week. Because respondents describe full time has 35+ hours a week, this potentially excludes workers making up to 8.29 per hour but only working 35 hours per week. These workers are unlikely to be affected significantly by EPEWA, so this is considered an acceptable exclusion.

The combination of (4) and (5) above result in the data considering approximately the middle 60% of the U.S. income distribution, with a slight skew to higher earners.

In [11]:
def check_window(df):
    """Stop if the loaded data does not cover exactly the window set in the config cell."""
    ym = df['YEAR'] * 100 + df['MONTH']
    expected = (SAMPLE_START[0] * 100 + SAMPLE_START[1], SAMPLE_END[0] * 100 + SAMPLE_END[1])
    assert (ym.min(), ym.max()) == expected, (
        f"Loaded data covers {ym.min()} to {ym.max()}, not {expected[0]} to {expected[1]}. "
        "The newest file in DATA_DIR is for a different window: set SUBMIT_NEW_EXTRACT = True.")

# Cache of the eligible sample with NO upper earnings cut ("wide"). The name includes the sample window, so changing
# SAMPLE_START/SAMPLE_END forces a fresh load instead of silently reusing old data.
WIDE_CACHE = DATA_DIR / f"ipums_wide_{SAMPLE_START[0]}-{SAMPLE_START[1]:02d}_{SAMPLE_END[0]}-{SAMPLE_END[1]:02d}.pkl"

if WIDE_CACHE.exists():
    wide = pd.read_pickle(WIDE_CACHE)
    check_window(wide)
    print(f"Loaded cached data from {WIDE_CACHE.name}")
else:
    iter_microdata = readers.read_microdata_chunked(ddi, dat_file, chunksize=3000000)

    # Filters: MISH=8 (unique earnings per person), full-time workers (WKSTAT=11), ages 25-54, weekly earnings above
    # $290 (the federal minimum wage for 40 hours). N.I.U. (999999.99) is removed. There is no upper earnings cut here:
    # it is applied afterwards so Section 6 can compare alternatives on the same sample.
    wide = pd.concat([
        chunk[
            (chunk["MISH"] == 8) & (chunk["WKSTAT"] == 11) &
            (chunk["AGE"] >= 25) & (chunk["AGE"] <= 54) &
            (chunk["EARNWEEK2"] > 290) & (chunk["EARNWEEK2"] < 999999)
        ]
        for chunk in iter_microdata
    ], ignore_index=True)

    check_window(wide)                        # check BEFORE caching, so a wrong file never poisons the cache
    wide.to_pickle(WIDE_CACHE)
    print(f"Saved cache to {WIDE_CACHE.name}")

Loaded cached data from ipums_wide_2010-01_2026-09.pkl


In [12]:
# Main analysis: keep workers below the topcode (the "nominal cut"). Section 6 compares alternative cuts.
ipums_df = wide[wide['EARNWEEK2'] < TOPCODE_WEEKLY].copy()
print(f"{len(wide):,} eligible workers; {len(ipums_df):,} below the ${TOPCODE_WEEKLY:,.2f} weekly cut")

634,084 eligible workers; 599,085 below the $2,884.61 weekly cut


In [13]:
# Set display options for future results. This swaps scientific notation for 4 decimal places.

pd.set_option('display.float_format', '{:.4f}'.format)

# Section 2: Minimum Wage Webscrape and Exclusion

The following section builds a table of state minimum wages by year from three sources, then uses it to exclude from the sample workers who reported making less than a state's full time minimum wage weekly earnings in a given state in a given year:

1. The __[historical](https://www.dol.gov/agencies/whd/state/minimum-wage/history)__ US DOL table of state minimum wages, which reports the January 1 rate for each year from 2010 through 2024.
2. The __[current](https://www.dol.gov/agencies/whd/mw-consolidated)__ US DOL consolidated table, a snapshot with a single effective date stated in the table caption (July 1, 2026 as of this run). The code reads the date from the page and uses this table for 2026.
3. **2025, from Wikipedia (provisional).** DOL does not currently publish 2025 rates, so the 2025 column is taken from __[Wikipedia's table of state minimum wages](https://en.wikipedia.org/wiki/List_of_U.S._states_by_minimum_wage)__. This is not an official source. It is flagged with a TODO in the code and should be replaced with official values before any 2025 results are relied upon.

This exclusion is intended to address omitted variable bias of minimum wage increases reducing the gender wage gap. Because women earn less than men, it is likely that women are more likely to work minimum wage than men and benefit more from minimum wage increases than men.

This effect can differ significantly by state. In California, the minimum wage more than doubled between 2010 and 2026 ($8.00 to $16.90). Meanwhile in approximately 20 other states, the federal minimum wage remained constant during this time.

Where the US DOL posts a range of wages for a state minimum, the code assumes the highest of the range as the minimum wages.
Where there is no state minimum wage posted, the code assumes the federal minimum wage.

Once the minimum wage values are collected, they are multiplied by 40 hours to give a weekly threshold. The thresholds are merged with the IPUMS dataframe by state and year, and the exclusion is applied.

**Timing convention:** DOL's historical table reports January 1 rates, so every year through 2024 uses its January 1 rate. 2026 uses the July 1, 2026 snapshot, which may overstate the threshold for January to June in the few jurisdictions that change rates mid-year (for example Alaska, DC and Oregon). The original July 2024 analysis used a snapshot taken in July 2024 for 2024; switching to the January rate changes 26 observations, and the regression coefficients by 0.0001 or less.

**A note on methodology:** This approach only excludes those making less than the minimum wage at 40 hours per week. It is likely that minimum wage increases affect those making slightly above minimum wage by applying wage pressures with those with skill premiums slightly above minimum wage workers but participating in similar work. Sometimes this accounted for by excluding those making more than a certain amount above minimum wage (such as $1 more). In the source data, full-time work is classified as 35 hours per week or more, meaning there could be workers in the existing sample working 114% of minimum wage and working 35 hours per week that would be excluded. This is intended to capture those near the minimum wage, but the original approach of excluding a fixed dollar amount or fixed percentage above the minimum wage may be a superior approach.

In [14]:
# Scrape the DOL historical table of state minimum wages (January 1 rates, one column per year).
url = 'https://www.dol.gov/agencies/whd/state/minimum-wage/history'

response = requests.get(url, timeout=30)
response.raise_for_status()
soup = BeautifulSoup(response.text, 'html.parser')
tables = soup.find_all('table', class_='minwage')
assert tables, "No minimum wage tables found. The DOL page layout may have changed."

# The page splits the history into several tables, one per block of years. Read each into a dataframe.
table_frames = []
for table in tables:
    rows = table.find_all('tr')
    headers = [th.get_text() for th in rows[0].find_all('th')]
    data = [dict(zip(headers, [cell.text.strip() for cell in row.find_all(['td', 'th'])])) for row in rows[1:]]
    table_frames.append(pd.DataFrame(data))

# Combine the tables and tidy the columns: rename the state column and keep 2010 onward
concat_mw = pd.concat(table_frames, ignore_index=True)
concat_mw = concat_mw.rename(columns={'State or otherjurisdiction': 'State'})

pre_2010 = [col for col in concat_mw.columns if col != 'State' and col[:4].isdigit() and int(col[:4]) < 2010]
concat_mw = concat_mw.drop(columns=pre_2010)

# One row per jurisdiction (the tables repeat each state; take the first non-missing value in each column)
consol_mw = concat_mw.groupby('State').first().reset_index()

# DOL shows "..." where a state has no minimum of its own: use the federal minimum for that year
federal_row = consol_mw[consol_mw['State'] == 'Federal (FLSA)'].iloc[0]
for year in consol_mw.columns[1:]:
    consol_mw[year] = consol_mw[year].replace('...', federal_row[year])

print(f"Historical minimum wage table: {consol_mw.shape[0]} jurisdictions, years {consol_mw.columns[1]} to {consol_mw.columns[-1]}")

Historical minimum wage table: 55 jurisdictions, years 2010 to 2024


In [15]:
# Data cleaning function

def clean_wage_value(value):

    value_str = str(value)
    
    # Remove notes in parentheses or brackets
    
    if '(' in value_str or '[' in value_str:
        value_str = value_str.split('(')[0].split('[')[0]
    
    # Handle ranges by selecting the higher value
    
    if '-' in value_str:
        value_str = value_str.split('-')[-1]
    
    # Handle slashes by selecting the second (higher) value
    
    if '/' in value_str:
        value_str = value_str.split('/')[-1]
    
    # Format the value as a float, then as a formatted string
    
    try:
        formatted_value = "{:.2f}".format(float(value_str))
    except ValueError:
        # Return original value if conversion fails
        
        formatted_value = value
    
    return formatted_value

# Apply the cleaning function to all year columns

for col in consol_mw.columns[1:]:
    consol_mw[col] = consol_mw[col].apply(clean_wage_value)
    
# Remove territories from data

states_to_remove = ['Puerto Rico', 'U.S. Virgin Islands', 'Guam']
consol_mw = consol_mw[~consol_mw['State'].isin(states_to_remove)]

# Reset the index

consol_mw.reset_index(drop=True, inplace=True)

In [16]:
# Scrape the DOL "consolidated" table of CURRENT state minimum wages.
# This page is a snapshot for one effective date (stated in the table caption), not a fixed year,
# so the year is read from the page instead of being assumed.
url = 'https://www.dol.gov/agencies/whd/mw-consolidated'
response = requests.get(url, timeout=30)
response.raise_for_status()
soup = BeautifulSoup(response.text, 'html.parser')
table = soup.find('table', summary="consolidated minimum wage rates.")

# Effective date from the caption, e.g. "(Effective Date: July 1, 2026)"
caption = table.find('caption').get_text(' ', strip=True)
effective = datetime.strptime(re.search(r'Effective Date:\s*([A-Za-z]+ \d{1,2}, \d{4})', caption).group(1), '%B %d, %Y')
fed_wage = float(re.search(r'federal MW of \$(\d+(?:\.\d+)?)', table.get_text(' ', strip=True)).group(1))

# State cells look like "CA $16.90" or "NY $17.00 or $16.00"; where several wages are listed, take the highest.
current_wages = {}
for cell in table.find_all('td'):
    text = cell.get_text(' ', strip=True)
    if '$' not in text:
        continue
    state = text.split('$')[0].strip()
    current_wages[state] = max(float(a) for a in re.findall(r'\$(\d+(?:\.\d+)?)', text))

print(f"DOL snapshot effective {effective:%B %d, %Y}: {len(current_wages)} jurisdictions, federal MW ${fed_wage}")

# Crosswalk acronymns to full length state names

state_acronym_to_full = {
    'AL': 'Alabama', 'AK': 'Alaska', 'AZ': 'Arizona', 'AR': 'Arkansas', 'CA': 'California',
    'CO': 'Colorado', 'CT': 'Connecticut', 'DE': 'Delaware', 'FL': 'Florida', 'GA': 'Georgia',
    'HI': 'Hawaii', 'ID': 'Idaho', 'IL': 'Illinois', 'IN': 'Indiana', 'IA': 'Iowa',
    'KS': 'Kansas', 'KY': 'Kentucky', 'LA': 'Louisiana', 'ME': 'Maine', 'MD': 'Maryland',
    'MA': 'Massachusetts', 'MI': 'Michigan', 'MN': 'Minnesota', 'MS': 'Mississippi', 'MO': 'Missouri',
    'MT': 'Montana', 'NE': 'Nebraska', 'NV': 'Nevada', 'NH': 'New Hampshire', 'NJ': 'New Jersey',
    'NM': 'New Mexico', 'NY': 'New York', 'NC': 'North Carolina', 'ND': 'North Dakota', 'OH': 'Ohio',
    'OK': 'Oklahoma', 'OR': 'Oregon', 'PA': 'Pennsylvania', 'RI': 'Rhode Island', 'SC': 'South Carolina',
    'SD': 'South Dakota', 'TN': 'Tennessee', 'TX': 'Texas', 'UT': 'Utah', 'VT': 'Vermont',
    'VA': 'Virginia', 'WA': 'Washington', 'WV': 'West Virginia', 'WI': 'Wisconsin', 'WY': 'Wyoming',
    'DC': 'District of Columbia', 'PR': 'Puerto Rico', 'GU': 'Guam', 'VI': 'Virgin Islands', 'CNMI': 'Northern Mariana Islands',
    'AS': 'American Samoa'
}

current_wages_full = {state_acronym_to_full[k]: v for k, v in current_wages.items() if k in state_acronym_to_full}

DOL snapshot effective July 01, 2026: 34 jurisdictions, federal MW $7.25


In [17]:
# =====================================================================================================
# TODO (COME BACK AND FIX): 2025 minimum wages are taken from WIKIPEDIA, which is NOT an official source.
#
# Why: the DOL history table stops at 2024, and the DOL consolidated page is a July 2026 snapshot, so DOL
# currently publishes no 2025 values. This cell is a stopgap so the latest-data run can proceed.
#
# Before relying on any result that includes 2025:
#   1. Replace these values with official ones (DOL's 2025 consolidated table when it is posted, or each
#      state's labor department) and delete this cell.
#   2. Check the states with more than one rate (NJ, NY, OR). The code takes the HIGHEST rate listed,
#      matching the DOL rule used elsewhere, but Wikipedia lists only the standard rate for some states
#      (for example Oregon 2025), so the highest local rate may be missing.
# =====================================================================================================
WIKI_URL = 'https://en.wikipedia.org/wiki/List_of_U.S._states_by_minimum_wage'
NEEDS_2025 = 2025 in range(SAMPLE_START[0], SAMPLE_END[0] + 1)

def scrape_wikipedia_wages(year):
    """Return {state name: highest hourly minimum wage listed} from Wikipedia's state minimum wage table."""
    resp = requests.get(WIKI_URL, timeout=30, headers={'User-Agent': 'Mozilla/5.0 (research notebook)'})
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')
    for table in soup.find_all('table', class_='wikitable'):
        header = [c.get_text(' ', strip=True) for c in table.find('tr').find_all(['th', 'td'])]
        cols = [i for i, h in enumerate(header) if h.split()[:1] == [str(year)]]   # e.g. "2025" (ignores footnote marks)
        if header[:1] != ['State'] or not cols:
            continue
        wages = {}
        for row in table.find_all('tr')[1:]:
            cells = [c.get_text(' ', strip=True) for c in row.find_all(['th', 'td'])]
            state = re.sub(r'[\*†\s]+$', '', cells[0]).replace('Washington, D.C.', 'District of Columbia')
            amounts = [float(a) for a in re.findall(r'\$(\d+(?:\.\d+)?)', cells[cols[0]])]
            if amounts:
                wages[state] = max(amounts)
        return wages
    raise ValueError(f"No Wikipedia table with a {year} column found. The page layout may have changed.")

wiki_wages_2025 = scrape_wikipedia_wages(2025) if NEEDS_2025 else {}

if NEEDS_2025:
    print("WARNING: 2025 minimum wages come from Wikipedia (unofficial). See the TODO at the top of this cell.")
    print(f"{len(wiki_wages_2025)} jurisdictions found.")
    print({s: wiki_wages_2025[s] for s in ['California', 'Colorado', 'Washington', 'New Jersey', 'New York', 'Oregon', 'District of Columbia']})
else:
    print("2025 is outside the sample window, so the Wikipedia values are not needed.")

51 jurisdictions found.
{'California': 16.5, 'Colorado': 14.81, 'Washington': 16.66, 'New Jersey': 15.49, 'New York': 16.5, 'Oregon': 14.7, 'District of Columbia': 17.5}


In [18]:
# Assemble ONE minimum wage table covering every year in the study window.
# Sources, in priority order:
#   1. DOL history table (consol_mw): official, one column per year (through 2024 as of this writing)
#   2. DOL consolidated snapshot (current_wages_full): used only for the year of its effective date
#   3. TEMPORARY: Wikipedia values for 2025 (see the TODO cell above; replace with an official source)
needed_years = list(range(SAMPLE_START[0], SAMPLE_END[0] + 1))
year_columns = [str(y) for y in needed_years]

# 1. Make the history table's year columns numeric
for col in [c for c in consol_mw.columns if c.isdigit()]:
    consol_mw[col] = consol_mw[col].replace(r'[\$,]', '', regex=True).astype(float)

# 2. Current snapshot: states not listed on the page use the federal minimum
snap_year = str(effective.year)
if effective.year in needed_years and snap_year not in consol_mw.columns:
    consol_mw[snap_year] = consol_mw['State'].map(current_wages_full).fillna(fed_wage)

# 3. TEMPORARY: 2025 from Wikipedia. States at the federal minimum are listed there too; the Federal row gets fed_wage.
if NEEDS_2025 and '2025' not in consol_mw.columns:
    consol_mw['2025'] = consol_mw['State'].map(wiki_wages_2025).fillna(fed_wage)

# Every year in the window needs a wage for every state, or the run stops here.
missing_years = [y for y in year_columns if y not in consol_mw.columns]
assert not missing_years, f"No minimum wage data for {missing_years}."
assert not consol_mw[year_columns].isna().any().any(), "Some state-years have no minimum wage."

print(consol_mw.loc[consol_mw['State'].isin(['California','Colorado','Washington','Federal (FLSA)']), ['State'] + year_columns[-3:]])

             State    2024    2025    2026
4       California 16.0000 16.5000 16.9000
5         Colorado 14.4200 14.8100 15.1600
9   Federal (FLSA)  7.2500  7.2500  7.2500
48      Washington 16.2800 16.6600 17.1300


In [19]:
# Assertion check to ensure the minimum wages are never declining for a state
# Initialize a list to keep track of states with decreasing wage values

states_with_decreases = []

# Iterate over each row in the DataFrame

for index, row in consol_mw.iterrows():
    
    # Check each year's wage against the previous year's wage
    
    for i in range(1, len(year_columns)):
        previous_year = year_columns[i-1]
        current_year = year_columns[i]
   
        if pd.notna(row[current_year]) and pd.notna(row[previous_year]) and (float(row[current_year]) < float(row[previous_year])):
            states_with_decreases.append((row['State'], current_year))
            break

if states_with_decreases:
    print("States with at least one year where the minimum wage decreased from the previous year:")
    for state, year in states_with_decreases:
        print(f"State: {state}, Year: {year}")
else:
    print("All states have minimum wages equal to or greater than the previous year for every year.")


All states have minimum wages equal to or greater than the previous year for every year.


In [20]:
# Define the statefips mapping dictionary
state_name_to_fips = {
    'Federal (FLSA)': '00',
    'Alabama': '01',
    'Alaska': '02',
    'Arizona': '04',
    'Arkansas': '05',
    'California': '06',
    'Colorado': '08',
    'Connecticut': '09',
    'Delaware': '10',
    'District of Columbia': '11',
    'Florida': '12',
    'Georgia': '13',
    'Hawaii': '15',
    'Idaho': '16',
    'Illinois': '17',
    'Indiana': '18',
    'Iowa': '19',
    'Kansas': '20',
    'Kentucky': '21',
    'Louisiana': '22',
    'Maine': '23',
    'Maryland': '24',
    'Massachusetts': '25',
    'Michigan': '26',
    'Minnesota': '27',
    'Mississippi': '28',
    'Missouri': '29',
    'Montana': '30',
    'Nebraska': '31',
    'Nevada': '32',
    'New Hampshire': '33',
    'New Jersey': '34',
    'New Mexico': '35',
    'New York': '36',
    'North Carolina': '37',
    'North Dakota': '38',
    'Ohio': '39',
    'Oklahoma': '40',
    'Oregon': '41',
    'Pennsylvania': '42',
    'Rhode Island': '44',
    'South Carolina': '45',
    'South Dakota': '46',
    'Tennessee': '47',
    'Texas': '48',
    'Utah': '49',
    'Vermont': '50',
    'Virginia': '51',    
    'Washington': '53',
    'West Virginia': '54',
    'Wisconsin': '55',
    'Wyoming': '56'
}

# Use map function to create a new column 'statefip'

consol_mw['STATEFIP'] = consol_mw['State'].map(state_name_to_fips)

# Check for states that didn't match the mapping (Debugging step)

unmapped_states = consol_mw[consol_mw['STATEFIP'].isna()]['State'].unique()
print("Unmapped States:", unmapped_states)

Unmapped States: []


In [21]:
# Confirm size of IPUMS dataframe before excluding those making less than minimum wage in a given year / state

print(len(ipums_df))

599085


In [22]:
def apply_min_wage_exclusion(df, wages):
    """Drop workers whose weekly earnings are at or below 40 hours x their state's minimum wage for their year.
    `wages` has one row per state (column 'State') and one column per year ('2010', '2011', ...)."""
    long = wages.melt(id_vars='State', var_name='YEAR', value_name='min_wage')
    long['YEAR'] = long['YEAR'].astype(int)
    long['STATEFIP'] = long['State'].map(state_name_to_fips)
    merged = df.merge(long[['STATEFIP', 'YEAR', 'min_wage']], on=['STATEFIP', 'YEAR'], how='left')
    assert merged['min_wage'].notna().all(), "Some workers have no minimum wage for their state and year."
    kept = merged[merged['EARNWEEK2'] > 40 * merged['min_wage']]
    return kept.drop(columns='min_wage').reset_index(drop=True)

In [23]:
# Start from the wide sample every time, so this cell is safe to rerun
ipums_df = wide[wide['EARNWEEK2'] < TOPCODE_WEEKLY].copy()
ipums_df['STATEFIP'] = ipums_df['STATEFIP'].astype(str).str.pad(width=2, fillchar='0')
print("Rows before exclusion:", len(ipums_df))

ipums_df = apply_min_wage_exclusion(ipums_df, consol_mw[['State'] + year_columns])
print("Rows after exclusion:", len(ipums_df))

Rows before exclusion: 599085
Rows after exclusion: 588770


Approximately 10,300 observations (10,315 of 599,085) were excluded for respondents making below the minimum wage for their state and year.

# Section 3: Regression Variables and Controls Preparation

Before proceeding to the regressions, additional variables beyond female need to be generated for other considerations. These variables and their explanations are:

**Comparison Categories**: The primary finding is to evaluate whether EPEWA had an impact on the gender wage gap in Colorado. More specifically, whether the EPEWA requirement to include intended pay ranges in job postings reduced the difference in earnings between men and women. To evaluate this, Colorado is compared to the non pay transparent states. Since EPEWA came into effect in 2021, two other states passed similar laws in a time frame that allows for comparisons. California and Washington both had laws come into effect in January 2023. These laws were less stringent than EPEWA, applying only to employers with 15 employees or more where EPEWA applied to all employers. Because EPEWA was in effect for two more years (2021 and 2022) and applies to a wider universe of employers, it is anticipated that the reduction in the gender wage gap for Colorado will be greater than Washington, California, and the remaining 47 states and DC. **For California and Washington the post-law period starts in January 2023, the month their own laws took effect (variable `epewa_own`); every other state, including Colorado, uses January 2021.** The group comparisons in Section 4, the state regressions in Section 5 and the Section 6 robustness regressions all use this timing. Because California and Washington passed similar laws with less time and less applicability, they are considered to have a weaker treatment effect and it is anticipated that they will have a reduction in the gender wage gap between Colorado and the 47 states and DC. Therefore for the comparisons, a categorical variable with Colorado, Washington, California, and the remaining 47 states and DC is generated. Note: Hawaii passed a similar pay transparency law that came into effect January 2024. Given this represents such a short time in the sample, Hawaii is included with the non pay transparent states. The City of New York is treated similarly for the law that came into effect in November 2022 - it is hard to isolate the geography of New York City in the data as the CPS groups it with Newark. Since this analysis was first written, the number of states with pay transparency laws has grown to 14. Several of those states are still in the comparison group here, which makes it a less clean control as the sample extends to 2026 and is likely to understate the difference between Colorado and the rest of the country. Hawaii's law took effect in January 2024 and now covers more than two years of the sample, so it is a candidate for separate treatment. An extended analysis that treats the newer states separately is planned.

**Annual Earnings**: The EARNWEEK2 variable is multiplied by 52 to approximately annual earnings for easier comprehension. It is logarithmically transformed for later regression analysis, so coefficients can be read as approximate percentage differences.

**EPEWA**: Defining the period after EPEWA came into effect. There is additional analysis that could be considered that evaluates lags and leads of pay transparency laws coming into effect. As this analysis stands, a simple before / after comparison is used, with the break at January 2021 when EPEWA came into effect (January 2023 for California and Washington, when their laws took effect). This is due to other research showing much of the impact in jobs postings including pay ranges occuring in Colorado in the early months of 2021. The post-law period now runs from January 2021 through September 2026, so the EPEWA coefficient also absorbs nominal wage growth over a longer window.

In some of the regression analysis, EPEWA in isolation is used to capture wage inflation effects since January 2021.

**Race Categories**: This is to control by race. When measuring the gender wage gap, there is well established intersectionality of wage penalties possibly due to discrimination. In other words there are three known effects that can explain differences in earnings:

1. Workers tend to be paid less if they are a woman.
2. Workers tend to be paid less if they are a person of color.
3. Workers tend to be paid less if they are a woman of color.

When measuring the gender wage gap (1) adding a control for race ensures the analysis is not measuring (3). This controls for omitted variable bias in Colorado, Washington, or California having different racial make ups than the rest of the United States explaining the difference in gender wage gaps.

For this analysis, hispanic is considered a race. Categorization for __[race](https://cps.ipums.org/cps-action/variables/RACE#codes_section)__ and __[hispanic](https://cps.ipums.org/cps-action/variables/HISPAN#codes_section)__
were sourced from IPUMS.

**Industry**: Another possible explanation for the difference in earnings between men and women is that women tend to enter lower paying industries than men. Whether this behavior is due to discrimination against women in the form of societal conditioning or due to innate preferences of women is out of the scope of this analysis. Therefore this analysis considers both.

In court cases for wage discrimination, where components of EPEWA apply, measurements of damages often rely on within-industry wage comparisons. Therefore this analysis considers industry controls the preferred specification.

The industry categorization comes from __[this IPUMS crosswalk](https://usa.ipums.org/usa/volii/indtoindnaics18.shtml)__ of the CPS 'IND' variable to 2-digit NAICS code industries. The crosswalk has three periods for the years considered (2008-2012, 2013-2017 and 2018-2022). The code finds these columns by pattern, since IPUMS has relabeled them before, and uses the period that contains each worker's survey year. The CPS adopted the 2022 Census industry classification in January 2025, and IPUMS has not published a crosswalk for it. The 34 new codes are therefore assigned to the sector of the nearest lower code in the crosswalk, using IPUMS's own list of 2025-onward codes. The assignments are printed in a table in the code and were checked by hand, but this is a provisional bridge, flagged with a TODO, to be replaced when an official crosswalk exists.

**Age**: Age is controlled for as a proxy for experience. While aging does not necessarily result in experience accumulation, it is impossible to accumulate experience without aging. Controlling for age accounts for possible differences in experience amongst the two male and female sub samples.

**Education**: Education is another determinant of income that may result in omitted variable bias. Men and women have differing rates of educational attaintment. Men have historically had higher levels of education, but this trend has reversed in more recent years as women make a majority of college enrollments. Controlling for education ensures the analysis does not measure these differing rates of education. The educational categorizations come from __[here](https://cps.ipums.org/cps-action/variables/EDUC#codes_section)__.

In [24]:
# Generate state-specific dummy variables for states with pay transparency active as of Jan 2023.

ipums_df['co'] = (ipums_df['STATEFIP'] == '08')
ipums_df['ca'] = (ipums_df['STATEFIP'] == '06')
ipums_df['wa'] = (ipums_df['STATEFIP'] == '53')

# Use CO, WA, and CA variables to create a single comparison state variable

conditions = [
    ipums_df['ca'] == 1,
    ipums_df['co'] == 1,
    ipums_df['wa'] == 1
]
choices = [1, 2, 3]  # 1 for California, 2 for Colorado, 3 for Washington
ipums_df['comp_cat4'] = np.select(conditions, choices, default=0)

# Convert to a categorical variable with labels

comp_cat4_labels = {0: 'Rest of US', 1: 'California', 2: 'Colorado', 3: 'Washington'}
ipums_df['comp_cat4'] = ipums_df['comp_cat4'].map(comp_cat4_labels).astype('category')

# Display the distribution of the categorical variable

display(ipums_df['comp_cat4'].value_counts(dropna=False))

comp_cat4
Rest of US    521535
California     47168
Washington     10122
Colorado        9945
Name: count, dtype: int64

In [25]:
# Generate annual earnings variable. This makes results easier to conceptualize over weekly earnings.

ipums_df['earnyear'] = ipums_df['EARNWEEK2'] * 52

# Generate log annual earnings variable. This will allow for percentage point comparisons for most of the regressions.

ipums_df['log_earnyear'] = np.log(ipums_df['earnyear'])

In [26]:
# Create variable for when Equal Pay for Equal Work Act is in effect

ipums_df['epewa'] = (ipums_df['YEAR'] >= 2021).astype(int)

In [27]:
# Each state's own post-law period. California's and Washington's pay transparency laws took effect in January 2023.
# Colorado's took effect in January 2021, and 2021 is also the reference date for every other state.
LAW_START_YEAR = {'06': 2023, '53': 2023}          # STATEFIP codes: California, Washington
ipums_df['epewa_own'] = (ipums_df['YEAR'] >= ipums_df['STATEFIP'].map(LAW_START_YEAR).fillna(2021)).astype(int)

In [28]:
# Create binary gender variable

ipums_df['female'] = (ipums_df['SEX'] == 2).astype(int)

In [29]:
# Create binary variables for race categories including Hispanic. 
# Hispanic is an ethnicity, not a race, but is useful for grouping by race.

ipums_df['hisp'] = (ipums_df['HISPAN'] >= 100).astype(int)
ipums_df['white'] = ((ipums_df['RACE'] == 100) & (ipums_df['hisp'] == 0)).astype(int)
ipums_df['black'] = ((ipums_df['RACE'].isin([200, 801])) & (ipums_df['hisp'] == 0)).astype(int)
ipums_df['indian'] = ((ipums_df['RACE'].isin([300, 802])) & (ipums_df['hisp'] == 0)).astype(int)
ipums_df['asian'] = ((ipums_df['RACE'].isin([651, 803])) & (ipums_df['hisp'] == 0)).astype(int)
ipums_df['other_race'] = ((ipums_df['RACE'].isin([652]) | (ipums_df['RACE'] >= 804)) & (ipums_df['hisp'] == 0)).astype(int)

# Combine the binary flags into a single 'race_cat' variable with Hispanic included

conditions = [
    ipums_df['hisp'] == 1,
    ipums_df['white'] == 1,
    ipums_df['black'] == 1,
    ipums_df['indian'] == 1,
    ipums_df['asian'] == 1,
    ipums_df['other_race'] == 1,
]

# Define the race categories including Hispanic

choices = ['Hispanic', 'White', 'Black', 'Indian', 'Asian', 'Other Race']

# Assign race categories based on conditions

ipums_df['race_cat'] = np.select(conditions, choices, default='Not Specified')

# Print the distribution of the race category variable

race_cat_distribution = ipums_df['race_cat'].value_counts(dropna=False)
display(race_cat_distribution)

assert (ipums_df['race_cat'] != 'Not Specified').all(), "Some workers have an unrecognized RACE/HISPAN code."

race_cat
White         391357
Hispanic       87450
Black          60776
Asian          37176
Indian          7247
Other Race      4764
Name: count, dtype: int64

Note there are no observations that are not specified.

In [30]:
# To create industry categories, load the crosswalk from IPUMS.
# Downloaded with requests (which carries the network workaround from the setup section) and kept on disk.
# Delete the saved file to fetch a fresh copy if IPUMS posts a new version.
ipums_cw_link = 'https://usa.ipums.org/usa/resources/volii/ind_indnaics_crosswalk_2000_onward_without_code_descriptions.csv'
CW_FILE = DATA_DIR / 'ipums_ind_naics_crosswalk.csv'

if not CW_FILE.exists():
    resp = requests.get(ipums_cw_link, timeout=30)
    resp.raise_for_status()
    CW_FILE.write_bytes(resp.content)
    print(f"Downloaded the crosswalk ({len(resp.content) // 1024} KB)")

ipums_cw = pd.read_csv(CW_FILE)

In [31]:
def process_ind_codes(code_series):
    if pd.isna(code_series):
        return []
    # Handle float-formatted strings and possible whitespace in entries
    processed_codes = []
    for code in str(code_series).split(','):
        try:
            # Attempt to directly convert to int
            processed_code = int(code.strip())
        except ValueError:
            # If direct conversion fails, treat as float then convert to int
            processed_code = int(float(code.strip()))
        processed_codes.append(processed_code)
    return processed_codes

In [32]:
data_cleaned = ipums_cw.rename(columns={'Unnamed: 0': 'Category'})
data_cleaned['Category'] = data_cleaned['Category'].ffill().fillna('N/A')

# Find the IND CODE columns by pattern, not by exact name. IPUMS relabels periods
# (e.g. "2018 Onward" became "2018-2022"), which broke the hardcoded names.
# Columns look like "2013-2017 ACS/PRCS IND CODE"; the INDNAICS columns do not match.
period_cols = {}   # start year of the period -> column name
for col in data_cleaned.columns:
    m = re.search(r'(\d{4})(?:-\d{4}|\s+Onward)?\s+ACS/PRCS IND CODE\s*$', str(col))
    if m:
        period_cols[int(m.group(1))] = col
period_cols = {y: c for y, c in sorted(period_cols.items()) if y >= 2008}   # CPS samples start in 2010
print("IND code periods found:", period_cols)
assert period_cols, "No ACS/PRCS IND CODE columns found. IPUMS may have changed the crosswalk layout."

# One lookup per period: IND code -> industry category (first category wins, as before)
ind_lookup = {}
for start, col in period_cols.items():
    codes = data_cleaned[col].apply(process_ind_codes)
    lookup = {}
    for category, code_list in zip(data_cleaned['Category'], codes):
        for code in code_list:
            lookup.setdefault(code, category)
    ind_lookup[start] = lookup

IND code periods found: {2008: '2008-2012 ACS/PRCS IND CODE', 2013: '2013-2017 ACS/PRCS IND CODE', 2018: '2018-2022 ACS/PRCS IND CODE'}


In [33]:
# TODO (COME BACK AND FIX): IPUMS's ACS/PRCS crosswalk stops at the 2018-2022 code scheme, but the CPS switched to the
# 2022 Census classification in 2025. This BRIDGES the new 2025+ codes to sectors using the nearest lower code that
# is in the crosswalk (Census industry codes are numbered by sector). Replace it with an official crosswalk for the
# 2022 scheme when IPUMS posts one, and recheck the table below.
resp = requests.get('https://cps.ipums.org/cps/codes/ind_2025_codes.shtml', timeout=30)
resp.raise_for_status()
dl = BeautifulSoup(resp.text, 'html.parser').find('dl', class_='list-fixed-dt')
ind_2025 = {int(dt.get_text(strip=True)): dd.get_text(' ', strip=True) for dt, dd in zip(dl.find_all('dt'), dl.find_all('dd'))}

latest = max(ind_lookup)                         # newest crosswalk period (2018-2022)
known = sorted(ind_lookup[latest])
bridge = []
for code, desc in ind_2025.items():
    if code not in ind_lookup[latest]:
        i = bisect.bisect_right(known, code) - 1
        bridge.append({'IND': code, 'IPUMS description (2025+)': desc,
                       'nearest lower code': known[i] if i >= 0 else None,
                       'assigned category': ind_lookup[latest][known[i]] if i >= 0 else 'N/A'})
bridge_table = pd.DataFrame(bridge)
print(f"{len(ind_2025)} codes in the IPUMS 2025+ list; {len(bridge_table)} not in the crosswalk and bridged:")
display(bridge_table)

# A new "2025" period: everything in the 2018-2022 lookup plus the bridged codes
ind_lookup[2025] = {**ind_lookup[latest], **dict(zip(bridge_table['IND'], bridge_table['assigned category']))}

266 codes in the IPUMS 2025+ list; 34 not in the crosswalk and bridged:


,IND,IPUMS description (2025+),nearest lower code,assigned category
0,480,Not specified type of mining,470,"Mining, Quarrying, and Oil and Gas Extraction:"
1,4681,Other motor vehicle dealers,4680,Retail Trade:
2,4691,"Automotive parts, accessories, and tire retailers",4690,Retail Trade:
3,4771,Furniture and home furnishings retailers,4770,Retail Trade:
4,4796,Electronics and appliance retailers,4795,Retail Trade:
5,4871,"Building material and supplies dealers, except...",4870,Retail Trade:
6,4881,Hardware retailers,4880,Retail Trade:
7,4891,Lawn and garden equipment and supplies retailers,4890,Retail Trade:
8,4973,Convenience retailers and vending machine oper...,4972,Retail Trade:
9,4981,Specialty food retailers,4980,Retail Trade:


In [34]:
def lookup_category(year, ind_code):
    """Industry category for an IND code: use the period that contains the year, then fall back to the
    nearest other periods (ties go to the later period), as the original did."""
    primary = max((s for s in ind_lookup if s <= year), default=min(ind_lookup))
    order = [primary] + sorted((s for s in ind_lookup if s != primary), key=lambda s: (abs(s - primary), -s))
    for s in order:
        cat = ind_lookup[s].get(ind_code)
        if cat is not None:
            return cat
    return 'N/A'

ipums_df['ind_cat20'] = [lookup_category(y, c) for y, c in zip(ipums_df['YEAR'], ipums_df['IND'])]
ipums_df['ind_cat20_label'] = ipums_df['ind_cat20']

print("Count of 'N/A' in ind_cat20_label:", (ipums_df['ind_cat20_label'] == 'N/A').sum())

Count of 'N/A' in ind_cat20_label: 0


In [35]:
# Coverage check: unmatched workers by year. Watch the newest years, because the crosswalk
# only goes through 2018-2022 and newer IND codes may be unmapped.
unmatched = (ipums_df['ind_cat20'] == 'N/A').groupby(ipums_df['YEAR']).agg(['sum', 'mean'])
unmatched.columns = ['unmatched_rows', 'share']
print(unmatched.round(4))
assert unmatched['share'].max() < 0.01, "More than 1% of workers in some year have no industry match."

      unmatched_rows  share
YEAR                       
2010               0 0.0000
2011               0 0.0000
2012               0 0.0000
2013               0 0.0000
2014               0 0.0000
2015               0 0.0000
2016               0 0.0000
2017               0 0.0000
2018               0 0.0000
2019               0 0.0000
2020               0 0.0000
2021               0 0.0000
2022               0 0.0000
2023               0 0.0000
2024               0 0.0000
2025               0 0.0000
2026               0 0.0000


Note there are no observations in the 'N/A' category, meaning every observation was assigned.

In [36]:
# Convert AGE to a categorical variable named 'age'

ipums_df['age'] = ipums_df['AGE'].astype('category')

# Optionally, to name categories based on the integer value of AGE:

ipums_df['age'] = ipums_df['age'].cat.rename_categories(lambda x: f"Age {x}")

# Check the first few rows to confirm the conversion

age_distribution = ipums_df['age'].value_counts(dropna=False)
age_distribution = age_distribution.sort_index()
print(age_distribution)

age
Age 25    17805
Age 26    18796
Age 27    19518
Age 28    19651
Age 29    20350
Age 30    20672
Age 31    20975
Age 32    20574
Age 33    20522
Age 34    20019
Age 35    20087
Age 36    19887
Age 37    19679
Age 38    19427
Age 39    19434
Age 40    19669
Age 41    19769
Age 42    19305
Age 43    19222
Age 44    19085
Age 45    19269
Age 46    19408
Age 47    19266
Age 48    19506
Age 49    19394
Age 50    19597
Age 51    19971
Age 52    19396
Age 53    19434
Age 54    19083
Name: count, dtype: int64


In [37]:
# EDUC crosswalk

educ_crosswalk = {
    2: "None or preschool",
    10: "Grades 1, 2, 3, or 4",
    20: "Grades 5 or 6",
    30: "Grades 7 or 8",
    40: "Grade 9",
    50: "Grade 10",
    60: "Grade 11",
    71: "12th grade, no diploma",
    73: "High school diploma or equivalent",
    81: "Some college but no degree",
    91: "Associate's degree, occupational/vocational",
    92: "Associate's degree, academic program",
    111: "Bachelor's degree",
    123: "Master's degree",
    124: "Professional school degree",
    125: "Doctorate degree",
}


# Convert EDUC to a categorical variable named 'educ'

ipums_df['educ'] = ipums_df['EDUC'].map(educ_crosswalk).astype('category')

# Check the conversion by displaying the first few entries

education_distribution = ipums_df['educ'].value_counts(dropna=False)
education_distribution = education_distribution.sort_index()
print(education_distribution)

assert ipums_df['educ'].notna().all(), "Some workers have an EDUC code missing from educ_crosswalk."

educ
12th grade, no diploma                           4908
Associate's degree, academic program            39999
Associate's degree, occupational/vocational     30269
Bachelor's degree                              157657
Doctorate degree                                11520
Grade 10                                         4944
Grade 11                                         6648
Grade 9                                          5008
Grades 1, 2, 3, or 4                             1766
Grades 5 or 6                                    4775
Grades 7 or 8                                    4118
High school diploma or equivalent              150750
Master's degree                                 66119
None or preschool                                 717
Professional school degree                       7987
Some college but no degree                      91585
Name: count, dtype: int64


# Section 4: Primary Regression Analysis

The following section runs the primary regressions and produces the comparison charts.

The primary model uses a regression of the logarithmically transformed annual earnings as the endogenous (y) variable. It uses gender, the EPEWA time period, age, (sometimes) industry, race, and education as exogenous (x) variables.

It also uses the interaction term, female:epewa. This is the primary finding of the regressions. This is the reduction in the gender wage gap attributable to EPEWA and the other pay transparency laws using this difference in difference experimental design.

The code in this section fits the model separately for each comparison group (Colorado, Washington, California, and the rest of the country, as defined in Section 3), first without industry controls and then with them. All regressions use robust (HC3) standard errors. Full regression summaries for any group can be printed from the stored models, for example `models_ind['Colorado'].summary()`.

**Units.** The regression coefficients are in log points. To report the usual "cents on the dollar" the code converts them to exact ratios: women's earnings as a share of men's is exp(β) before the law and exp(β + γ) after it, where β is the female coefficient and γ the interaction. The change reported in the charts and text below is the difference between those two ratios (`ratio_change`). It is slightly smaller than the interaction coefficient itself (for Colorado, 9.0 log points but 7.5 cents) because the log-point coefficient is measured against the starting level.

In [38]:
models_no_ind, results_df = run_by_group(ipums_df, 'comp_cat4', FORMULA_NO_IND, post_col='epewa_own')
print('Regression Results for Key Variables by comp_cat4 Category, No Industry Controls')
display(results_df.round(4))

Regression Results for Key Variables by comp_cat4 Category, No Industry Controls


,female_coef,female_pval,interaction_coef,interaction_pval,interaction_se,pre_epewa_wage_gap,post_epewa_wage_gap,ratio_change
California,-0.1461,0.0000,0.0236,0.0105,0.0092,0.8641,0.8847,0.0206
Colorado,-0.2180,0.0000,0.0897,0.0000,0.0195,0.8041,0.8796,0.0755
Rest of US,-0.2146,0.0000,0.0180,0.0000,0.0026,0.8069,0.8215,0.0146
Washington,-0.2116,0.0000,0.0552,0.0053,0.0198,0.8093,0.8553,0.0460


In [39]:
fig = plot_group_gaps(results_df, "Pre- and Post-EPEWA Wage Gaps by Comparison Group", "Age, Education, and Race",
                      save_as="groups_no_industry")

The interaction coefficients are significant at the 5% level for all four comparison groups (the largest p-value is 0.0105, for California). Without industry controls, the change in women's earnings as cents on the dollar of men's is 7.5 in Colorado, 4.6 in Washington, 2.1 in California and 1.5 in the rest of the country.

Women's earnings in Colorado rose from 80.4 to 88.0 cents on the dollar of men's earnings after EPEWA came into effect, a larger change than in Washington (80.9 to 85.5), California (86.4 to 88.5) or the rest of the country (80.7 to 82.2). The relative sizes are consistent with the anticipated treatment effects. Washington and California are measured from January 2023, when their own laws took effect; measuring them from January 2021 instead would give them larger changes (6.2 and 2.7 log points rather than 5.5 and 2.4), but those years predate their laws.

In [40]:
models_ind, results_df = run_by_group(ipums_df, 'comp_cat4', FORMULA_IND, post_col='epewa_own')
print('Regression Results for Key Variables by comp_cat4 Category, Industry Controls')
display(results_df.round(4))

Regression Results for Key Variables by comp_cat4 Category, Industry Controls


,female_coef,female_pval,interaction_coef,interaction_pval,interaction_se,pre_epewa_wage_gap,post_epewa_wage_gap,ratio_change
California,-0.1327,0.0000,0.0234,0.0100,0.0091,0.8757,0.8965,0.0208
Colorado,-0.1778,0.0000,0.0834,0.0000,0.0190,0.8371,0.9100,0.0728
Rest of US,-0.1815,0.0000,0.0179,0.0000,0.0025,0.8340,0.8491,0.0151
Washington,-0.1683,0.0000,0.0503,0.0099,0.0195,0.8451,0.8886,0.0436


In [41]:
fig = plot_group_gaps(results_df, "Pre- and Post-EPEWA Wage Gaps by Comparison Group", "Age, Education, Race, and Industry",
                      save_as="groups_industry")

With industry controls, the change is 7.3 cents in Colorado, 4.4 in Washington, 2.1 in California and 1.5 in the rest of the country. All four are significant at the 5% level (the largest p-value is 0.0100, for California).

With industry controls, women's earnings in Colorado rose from 83.7 to 91.0 cents on the dollar of men's earnings, compared with 84.5 to 88.9 in Washington, 87.6 to 89.6 in California and 83.4 to 84.9 in the rest of the country. The relative values remain consistent with the anticipated treatment effects.

# Section 5: State Ranking and Charts

This section of code adds names to the dataset for each individual state (this code could be moved to section 3 but wasn't needed until this point).

To evaluate how the states rank against each other and the national average for the gender wage gap, the code then runs the regression model for each state and compiles the results into a dataframe. In this dataframe, the code converts the coefficients into the typical 'cents on the dollar' interpretation (women's earnings as a percentage of men's), using exact ratios: exp(β) before the law and exp(β + γ) after it. The change charts show the difference between the two. California and Washington use their own January 2023 start date; every other state uses January 2021.

An asterisk next to a state on a chart means the relevant coefficient is **not** statistically significant at the 10% level (p > 0.10): the pre-law gap for the first chart, and the change in the gap for the other two. The asterisks are applied by the code, not by hand. Across the 50 states and DC, 22 show a statistically significant change at the 10% level without industry controls and 23 with them (20 and 22 in the July 2024 analysis). At the 5% level the counts are 20 and 20 (14 and 16 previously).

This process is repeated with and without industry controls.

**Where Colorado ranks.** Before the law Colorado was 24th of the 50 states and DC without industry controls (22nd with them), close to the middle. After the law it ranks 3rd without industry controls (behind DC and California) and 2nd with them (behind DC). It has the largest increase in both specifications: 7.5 cents without industry controls and 7.3 with, ahead of South Dakota (5.3 and 5.6) and Alaska (5.0 and 5.4). Washington is fourth without industry controls (4.6 cents). For comparison, the national gap closed by 1.4 cents (1.5 with industry controls).

In [42]:
# Give state names to the STATEFIP codes (inverse of the name -> FIPS mapping built in Section 2)
fips_to_state = {fips: name for name, fips in state_name_to_fips.items() if name != 'Federal (FLSA)'}

ipums_df['STATEFIP'] = ipums_df['STATEFIP'].astype(str).str.pad(2, fillchar='0')
ipums_df['state'] = ipums_df['STATEFIP'].map(fips_to_state).astype('category')

assert ipums_df['state'].notna().all(), "Some STATEFIP codes are not in the state mapping."
display(ipums_df[['STATEFIP', 'state']].head())

,STATEFIP,state
0,01,Alabama
1,01,Alabama
2,01,Alabama
3,09,Connecticut
4,09,Connecticut


In [43]:
state_models_no_ind, results_df = run_by_group(ipums_df, 'state', FORMULA_NO_IND, include_national=True, post_col='epewa_own')
print('Individual State Regression Results for Key Variables, No Industry Controls')
display(results_df.round(4))

Individual State Regression Results for Key Variables, No Industry Controls


,female_coef,female_pval,interaction_coef,interaction_pval,interaction_se,pre_epewa_wage_gap,post_epewa_wage_gap,ratio_change
Alabama,-0.2507,0.0000,0.0302,0.0803,0.0173,0.7783,0.8022,0.0239
Alaska,-0.2292,0.0000,0.0607,0.0053,0.0218,0.7952,0.8450,0.0498
Arizona,-0.1683,0.0000,-0.0100,0.5946,0.0188,0.8451,0.8367,-0.0084
Arkansas,-0.2137,0.0000,0.0516,0.0046,0.0182,0.8076,0.8504,0.0428
California,-0.1461,0.0000,0.0236,0.0105,0.0092,0.8641,0.8847,0.0206
Colorado,-0.2180,0.0000,0.0897,0.0000,0.0195,0.8041,0.8796,0.0755
Connecticut,-0.1922,0.0000,0.0251,0.2605,0.0223,0.8252,0.8462,0.0210
Delaware,-0.1721,0.0000,0.0157,0.4872,0.0227,0.8419,0.8553,0.0134
District of Columbia,-0.0686,0.0000,0.0032,0.8365,0.0156,0.9337,0.9367,0.0030
Florida,-0.1713,0.0000,0.0247,0.0279,0.0112,0.8426,0.8636,0.0211


In [44]:
ranking_no_ind = results_df.rename(columns={'pre_epewa_wage_gap': 'Pre_EPEWA_Gap', 'post_epewa_wage_gap': 'Post_EPEWA_Gap'})
display(ranking_no_ind.round(4))

,female_coef,female_pval,interaction_coef,interaction_pval,interaction_se,Pre_EPEWA_Gap,Post_EPEWA_Gap,ratio_change
Alabama,-0.2507,0.0000,0.0302,0.0803,0.0173,0.7783,0.8022,0.0239
Alaska,-0.2292,0.0000,0.0607,0.0053,0.0218,0.7952,0.8450,0.0498
Arizona,-0.1683,0.0000,-0.0100,0.5946,0.0188,0.8451,0.8367,-0.0084
Arkansas,-0.2137,0.0000,0.0516,0.0046,0.0182,0.8076,0.8504,0.0428
California,-0.1461,0.0000,0.0236,0.0105,0.0092,0.8641,0.8847,0.0206
Colorado,-0.2180,0.0000,0.0897,0.0000,0.0195,0.8041,0.8796,0.0755
Connecticut,-0.1922,0.0000,0.0251,0.2605,0.0223,0.8252,0.8462,0.0210
Delaware,-0.1721,0.0000,0.0157,0.4872,0.0227,0.8419,0.8553,0.0134
District of Columbia,-0.0686,0.0000,0.0032,0.8365,0.0156,0.9337,0.9367,0.0030
Florida,-0.1713,0.0000,0.0247,0.0279,0.0112,0.8426,0.8636,0.0211


In [45]:
fig = plot_state_bars(ranking_no_ind, 'Pre_EPEWA_Gap', 'female_pval', "Women's Earnings As % of Men's",
                      "Age, Education, and Race", PRE_PERIOD, [0.7, 1], save_as="states_pre_gap_no_industry")

In [46]:
fig = plot_state_bars(ranking_no_ind, 'ratio_change', 'interaction_pval', "Change in Women's Earnings As % of Men's",
                      "Age, Education, and Race", f"{PRE_PERIOD} to {POST_PERIOD}", [-0.05, 0.1], save_as="states_change_no_industry")

In [47]:
fig = plot_state_bars(ranking_no_ind, 'Post_EPEWA_Gap', 'interaction_pval', "Women's Earnings As % of Men's",
                      "Age, Education, and Race", POST_PERIOD, [0.7, 1], save_as="states_post_gap_no_industry")

In [48]:
state_models_ind, results_df = run_by_group(ipums_df, 'state', FORMULA_IND, include_national=True, post_col='epewa_own')
print('Individual State Regression Results for Key Variables, Industry Controls')
display(results_df.round(4))

Individual State Regression Results for Key Variables, Industry Controls


,female_coef,female_pval,interaction_coef,interaction_pval,interaction_se,pre_epewa_wage_gap,post_epewa_wage_gap,ratio_change
Alabama,-0.2192,0.0000,0.0286,0.0955,0.0171,0.8031,0.8264,0.0233
Alaska,-0.1893,0.0000,0.0632,0.0030,0.0213,0.8276,0.8815,0.0540
Arizona,-0.1476,0.0000,-0.0068,0.7137,0.0185,0.8628,0.8569,-0.0059
Arkansas,-0.1879,0.0000,0.0498,0.0058,0.0181,0.8287,0.8711,0.0423
California,-0.1327,0.0000,0.0234,0.0100,0.0091,0.8757,0.8965,0.0208
Colorado,-0.1778,0.0000,0.0834,0.0000,0.0190,0.8371,0.9100,0.0728
Connecticut,-0.1765,0.0000,0.0259,0.2384,0.0219,0.8382,0.8601,0.0220
Delaware,-0.1601,0.0000,0.0221,0.3214,0.0223,0.8521,0.8711,0.0191
District of Columbia,-0.0568,0.0000,0.0056,0.7121,0.0151,0.9447,0.9500,0.0053
Florida,-0.1655,0.0000,0.0266,0.0164,0.0111,0.8474,0.8703,0.0229


In [49]:
ranking_ind = results_df.rename(columns={'pre_epewa_wage_gap': 'Pre_EPEWA_Gap', 'post_epewa_wage_gap': 'Post_EPEWA_Gap'})
display(ranking_ind.round(4))

,female_coef,female_pval,interaction_coef,interaction_pval,interaction_se,Pre_EPEWA_Gap,Post_EPEWA_Gap,ratio_change
Alabama,-0.2192,0.0000,0.0286,0.0955,0.0171,0.8031,0.8264,0.0233
Alaska,-0.1893,0.0000,0.0632,0.0030,0.0213,0.8276,0.8815,0.0540
Arizona,-0.1476,0.0000,-0.0068,0.7137,0.0185,0.8628,0.8569,-0.0059
Arkansas,-0.1879,0.0000,0.0498,0.0058,0.0181,0.8287,0.8711,0.0423
California,-0.1327,0.0000,0.0234,0.0100,0.0091,0.8757,0.8965,0.0208
Colorado,-0.1778,0.0000,0.0834,0.0000,0.0190,0.8371,0.9100,0.0728
Connecticut,-0.1765,0.0000,0.0259,0.2384,0.0219,0.8382,0.8601,0.0220
Delaware,-0.1601,0.0000,0.0221,0.3214,0.0223,0.8521,0.8711,0.0191
District of Columbia,-0.0568,0.0000,0.0056,0.7121,0.0151,0.9447,0.9500,0.0053
Florida,-0.1655,0.0000,0.0266,0.0164,0.0111,0.8474,0.8703,0.0229


In [50]:
fig = plot_state_bars(ranking_ind, 'Pre_EPEWA_Gap', 'female_pval', "Women's Earnings As % of Men's",
                      "Age, Education, Race, and Industry", PRE_PERIOD, [0.7, 1], save_as="states_pre_gap_industry")

In [51]:
fig = plot_state_bars(ranking_ind, 'ratio_change', 'interaction_pval', "Change in Women's Earnings As % of Men's",
                      "Age, Education, Race, and Industry", f"{PRE_PERIOD} to {POST_PERIOD}", [-0.04, 0.085], save_as="states_change_industry")

In [52]:
fig = plot_state_bars(ranking_ind, 'Post_EPEWA_Gap', 'interaction_pval', "Women's Earnings As % of Men's",
                      "Age, Education, Race, and Industry", POST_PERIOD, [0.7, 1], save_as="states_post_gap_industry")

# Section 6: Robustness to the Earnings Cut

The main analysis keeps workers earning less than $2,884.61 a week ($150,000 a year). That value is the CPS topcode for the outgoing rotation groups through March 2024, so the cut removes every censored earnings value. But it is fixed in nominal dollars, and wages have risen, so the cut removes a growing share of workers: about 3% in 2010 and 11% in 2026. Because Colorado, California and Washington have higher wages than most states, the cut can also affect them more. Dropping workers based on their earnings is also a form of selection on the outcome, so a cut that falls differently across states, genders and years can change the estimated effect. This section describes the problem and tests whether the main result survives alternative cuts.

**The topcode.** From April 2024 the Census Bureau replaced the fixed topcode with a monthly one for the top 3% of earners, so the censored share of this sample fell from up to about 12% to about 4%.

The wide sample loaded in Section 1 is the main sample before the upper cut. Four versions of the cut are compared, each applied after the minimum wage exclusion:

1. **Nominal cut (main analysis).** Weekly earnings below $2,884.61.
2. **CPI-deflated cap.** A $150,000 annual cap expressed in constant dollars of the last month of the sample, so the cut rises with prices. The CPI-U series is downloaded from the BLS when the section runs.
3. **Top 12% trimmed.** The top 12% of earners each month, with men and women pooled.
4. **Within-gender trim.** The top X% of earners within each gender each month, where X is the largest monthly share of men the nominal cut removes. This removes the same share of each gender in every month.

**Reading the results.** The comparison tables and charts report the change in women's earnings as cents on the dollar of men's after the law (the exact ratio change, exp(β + γ) − exp(β)), the same unit used in Sections 4 and 5. The tests of whether Colorado differs from the comparison groups and the placebo test below work directly with the regression coefficients, so they stay in log points (the coefficient times 100).

**What the results show.** The Colorado effect is positive under every cut, but its size depends on how the top of the earnings distribution is handled. Under the main nominal cut it is 7.5 cents without industry controls and 7.3 with; under the other three cuts it ranges from 4.3 to 5.5. The fixed nominal cut therefore gives the largest estimate, consistent with the concern that it removes a growing share of high earners in the later years. The difference between Colorado and the rest of the country ranges from 3.5 to 6.1 cents across the cuts and specifications, and it is statistically significant at the 5% level in every case (p-values between 0.000 and 0.024, from the test below). That test works with the coefficients directly and reports log points (4.6 to 7.2 without industry controls and 4.1 to 6.5 with), which is why its figures are larger than the cents above.

In [53]:
NOMINAL_CUT = TOPCODE_WEEKLY
SEX_LABEL = {1: 'Men', 2: 'Women'}
wide['above_cut'] = wide['EARNWEEK2'] >= NOMINAL_CUT

# 1. Share of eligible workers at or above the nominal cut, by year and sex (percent)
above = (wide.groupby(['YEAR', 'SEX'])['above_cut'].mean().unstack() * 100).rename(columns=SEX_LABEL)
above['All'] = wide.groupby('YEAR')['above_cut'].mean() * 100
above['Men as % of those above the cut'] = wide[wide['above_cut']].groupby('YEAR')['SEX'].apply(lambda s: (s == 1).mean() * 100)
print("Share of eligible full-time workers aged 25-54 at or above $2,884.61 a week (%)")
display(above.round(1))

# 2. Where the top of the distribution sits, by year (weekly dollars), and the cut as a percentile
pct = wide.groupby('YEAR')['EARNWEEK2'].quantile([0.5, 0.9, 0.95, 0.97, 0.99]).unstack()
pct.columns = [f"p{int(q * 100)}" for q in pct.columns]
pct['max'] = wide.groupby('YEAR')['EARNWEEK2'].max()
pct['cut as percentile'] = 100 - above['All']
print("Weekly earnings percentiles by year")
display(pct.round(0))

# 3. The topcode month by month: highest value and the share of workers sitting exactly at it
grp = wide.groupby(['YEAR', 'MONTH'])['EARNWEEK2']
monthly = pd.DataFrame({'n': grp.size(), 'max': grp.max()})
monthly['share_at_max'] = (wide['EARNWEEK2'] == grp.transform('max')).groupby([wide['YEAR'], wide['MONTH']]).mean() * 100
print("Monthly maximum earnings and the share of workers exactly at the maximum (2022 onward)")
display(monthly.reset_index().query('YEAR >= 2022').round(1).set_index(['YEAR', 'MONTH']))

Share of eligible full-time workers aged 25-54 at or above $2,884.61 a week (%)


SEX,Men,Women,All,Men as % of those above the cut
YEAR,,,,
2010,4.1000,1.6000,3.0000,76.3000
2011,4.0000,1.6000,3.0000,75.7000
2012,4.3000,1.8000,3.2000,75.7000
2013,4.8000,1.9000,3.5000,76.5000
2014,4.8000,2.0000,3.6000,75.3000
2015,5.2000,2.2000,3.9000,74.5000
2016,5.5000,2.3000,4.1000,74.9000
2017,5.8000,2.8000,4.5000,72.3000
2018,6.6000,3.2000,5.1000,72.3000


Weekly earnings percentiles by year


,p50,p90,p95,p97,p99,max,cut as percentile
YEAR,,,,,,,
2010,800.0000,1840.0000,2300.0000,2885.0000,2885.0000,2885.0000,97.0000
2011,808.0000,1850.0000,2360.0000,2880.0000,2885.0000,2885.0000,97.0000
2012,834.0000,1920.0000,2440.0000,2885.0000,2885.0000,2885.0000,97.0000
2013,850.0000,1920.0000,2500.0000,2885.0000,2885.0000,2885.0000,96.0000
2014,864.0000,1920.0000,2500.0000,2885.0000,2885.0000,2885.0000,96.0000
2015,866.0000,1980.0000,2600.0000,2885.0000,2885.0000,2885.0000,96.0000
2016,884.0000,2020.0000,2700.0000,2885.0000,2885.0000,2885.0000,96.0000
2017,922.0000,2120.0000,2760.0000,2885.0000,2885.0000,2885.0000,96.0000
2018,950.0000,2200.0000,2885.0000,2885.0000,2885.0000,2885.0000,95.0000


Monthly maximum earnings and the share of workers exactly at the maximum (2022 onward)


n        max  share_at_max
YEAR MONTH                               
2022 1      2758  2884.6000        7.5000
     2      2936  2884.6000        8.5000
     3      2788  2884.6000        8.6000
     4      2749  2884.6000        7.1000
     5      2927  2884.6000        7.7000
     6      2781  2884.6000        8.0000
     7      2609  2884.6000        8.4000
     8      2807  2884.6000        7.6000
     9      2850  2884.6000        7.8000
     10     2704  2884.6000        8.5000
     11     2562  2884.6000        7.7000
     12     2574  2884.6000        8.5000
2023 1      2676  2884.6000        9.9000
     2      2626  2884.6000        9.9000
     3      2606  2884.6000       10.0000
     4      2619  2884.6000        9.2000
     5      2643  2884.6000        9.3000
     6      2671  2884.6000        8.8000
     7      2640  2884.6000        9.5000
     8      2733  2884.6000        9.4000
     9      2635  2884.6000        9.8000
     10     2669  2884.6000        9.8000
     11     2550  2884.6000        8.8000
     12     2589  2884.6000        9.1000
2024 1      2720  2884.6000       11.7000
     2      2542  2884.6000       10.3000
     3      2408  2884.6000        9.6000
     4      2710  8686.1000        4.1000
     5      2716  7940.4000        3.9000
     6      2613 10772.8000        3.6000
     7      2563  8058.1000        4.2000
     8      2723 11952.5000        3.8000
     9      2894 10209.0000        3.1000
     10     2801  8962.7000        3.0000
     11     2753  8894.7000        4.7000
     12     2705  8947.5000        3.7000
2025 1      2833  9018.4000        4.2000
     2      2711  8115.1000        3.9000
     3      2666 10066.4000        3.8000
     4      2649  8686.4000        4.1000
     5      2551  9918.0000        3.5000
     6      2547 13145.8000        3.6000
     7      2329  6930.6000        4.6000
     8      2480  7785.9000        3.3000
     9      2702  9641.0000        3.8000
     11     2449 13258.2000        3.1000
     12     2460  7888.0000        3.9000
2026 1      2437 10838.6000        3.8000
     2      2476  9037.3000        3.3000
     3      2417  8916.2000        3.8000
     4      2524 10497.9000        4.1000
     5      2530  8962.0000        3.8000
     6      2311 11795.9000        4.2000
     7      2296 10779.6000        3.6000
     8      2496 10570.2000        3.3000
     9      2096 10579.7000        4.0000

In [54]:
from plotly.subplots import make_subplots

APR_2024 = '2024-04-01'   # the month the Census Bureau switched to a dynamic topcode


def style_explore(fig, title, subtitle, height=520):
    """Dark look shared with the other charts in this notebook."""
    fig.update_layout(
        title=dict(text=f"<b>{title}</b><br><span style='font-size:14px;color:#94a3b8'>{subtitle}</span>",
                   x=0.5, xanchor='center', y=1, yanchor='top', pad=dict(t=30), font=dict(size=22, color='#f8fafc')),
        font=dict(family=FONT, color='#cbd5e1'), paper_bgcolor='#0b1220', plot_bgcolor='#111a2e',
        height=height, margin=dict(l=80, r=50, t=130, b=70),
        legend=dict(orientation='h', x=0.5, xanchor='center', y=-0.15, yanchor='top', font=dict(size=13)))
    fig.update_xaxes(showgrid=False, tickfont=dict(size=13, color='#cbd5e1'), linecolor='rgba(148,163,184,0.4)')
    fig.update_yaxes(gridcolor='rgba(148,163,184,0.15)', zeroline=False, tickfont=dict(size=13, color='#cbd5e1'))


def mark_date(fig, x, label, color='#fbbf24'):
    """Vertical dashed line spanning the whole figure, with a label at the top."""
    fig.add_shape(type='line', x0=x, x1=x, y0=0, y1=1, xref='x', yref='paper', line=dict(color=color, width=2, dash='dash'))
    fig.add_annotation(x=x, y=1.0, xref='x', yref='paper', text=label, showarrow=False, xanchor='left', yanchor='bottom',
                       font=dict(size=12, color=color))


# ---- Figure 1: the share of workers the nominal cut removes -----------------------------------------------
fig1 = go.Figure()
for col, color, width, dash, pos, label_idx in [('Men', '#60a5fa', 3, 'solid', 'top center', (0, len(above) - 1)),
                                                ('Women', '#f472b6', 3, 'solid', 'bottom center', (0, len(above) - 1)),
                                                ('All', '#e2e8f0', 4, 'dot', 'middle right', (len(above) - 1,))]:
    labels = [f"{v:.1f}%" if i in label_idx else "" for i, v in enumerate(above[col])]
    fig1.add_trace(go.Scatter(x=above.index, y=above[col], name=col, mode='lines+markers+text', text=labels,
                              textposition=pos, textfont=dict(size=13, color=color),
                              line=dict(color=color, width=width, dash=dash), marker=dict(size=7),
                              hovertemplate='%{x}: %{y:.1f}%<extra>' + col + '</extra>'))
style_explore(fig1, "The fixed $2,884.61 cut removes a growing share of workers",
              "Share of eligible full-time workers aged 25-54 earning at or above the cut, by year")
fig1.update_xaxes(dtick=2, range=[2009.5, 2027.5])
fig1.update_yaxes(ticksuffix='%', title_text='% of workers removed')
mark_date(fig1, 2024.25, 'Dynamic topcode starts (Apr 2024)')
fig1.show()

# ---- Figure 2: the cut against the earnings distribution ---------------------------------------------------
fig2 = go.Figure()
for col, name, color in [('p50', 'Median', '#34d399'), ('p90', '90th percentile', '#60a5fa'), ('p95', '95th percentile', '#a78bfa')]:
    fig2.add_trace(go.Scatter(x=pct.index, y=pct[col], name=name, mode='lines+markers', line=dict(color=color, width=3),
                              marker=dict(size=6), hovertemplate='%{x}: $%{y:,.0f}<extra>' + name + '</extra>'))
fig2.add_hline(y=NOMINAL_CUT, line_dash='dash', line_color='#fbbf24', line_width=2,
               annotation_text=f"${NOMINAL_CUT:,.2f} cut", annotation_position='top left', annotation_font=dict(color='#fbbf24', size=13))
style_explore(fig2, "The distribution is moving up toward a cut that stays put",
              "Weekly earnings of eligible workers. The 95th percentile flattens at the cut where the topcode censors the data")
fig2.update_xaxes(dtick=2, range=[2009.5, 2026.8])
fig2.update_yaxes(tickprefix='$', tickformat=',', title_text='Weekly earnings')
mark_date(fig2, 2024.25, 'Dynamic topcode starts (Apr 2024)')
fig2.show()

# ---- Figure 3: the topcode, month by month ---------------------------------------------------------------
mm = monthly.reset_index()
mm['date'] = pd.to_datetime(dict(year=mm['YEAR'], month=mm['MONTH'], day=1))
fig3 = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12,
                     subplot_titles=("Highest weekly earnings value in the month", "Share of workers sitting exactly at that value"))
fig3.update_annotations(font=dict(size=14, color='#cbd5e1'))     # subplot titles only (no other annotations exist yet)
fig3.add_trace(go.Scatter(x=mm['date'], y=mm['max'], mode='lines+markers', line=dict(color='#34d399', width=3), marker=dict(size=5),
                          showlegend=False, hovertemplate='%{x|%b %Y}: $%{y:,.0f}<extra></extra>'), row=1, col=1)
fig3.add_trace(go.Scatter(x=mm['date'], y=mm['share_at_max'], mode='lines+markers', line=dict(color='#f472b6', width=3), marker=dict(size=5),
                          showlegend=False, hovertemplate='%{x|%b %Y}: %{y:.1f}%<extra></extra>'), row=2, col=1)
fig3.add_hline(y=NOMINAL_CUT, row=1, col=1, line_dash='dash', line_color='#fbbf24', annotation_text='$2,884.61',
               annotation_position='top left', annotation_font=dict(color='#fbbf24', size=12))
fig3.add_hline(y=12, row=2, col=1, line_dash='dot', line_color='#cbd5e1', annotation_text='12%',
               annotation_position='top left', annotation_font=dict(color='#cbd5e1', size=12))
style_explore(fig3, "The topcode changed in April 2024",
              "A fixed $2,884.61 topcode before, a monthly topcode for the top 3% of earners after",
              height=760)
mark_date(fig3, APR_2024, 'Apr 2024')
fig3.update_yaxes(tickprefix='$', tickformat=',', row=1, col=1)
fig3.update_yaxes(ticksuffix='%', row=2, col=1)
fig3.show()

# ---- Figure 4: the shape of the distribution around the cut, three years -------------------------------------
years_to_plot = [2010, 2023, 2026]
colors4 = ['#60a5fa', '#34d399', '#f472b6']
bins = np.arange(0, 6100, 100)
fig4 = make_subplots(rows=1, cols=3, shared_yaxes=True, horizontal_spacing=0.03,
                     subplot_titles=[str(y) for y in years_to_plot])
fig4.update_annotations(font=dict(size=14, color='#cbd5e1'))     # subplot titles only (no other annotations exist yet)
for i, (yr, color) in enumerate(zip(years_to_plot, colors4), start=1):
    e = wide.loc[wide['YEAR'] == yr, 'EARNWEEK2']
    counts, edges = np.histogram(e, bins=bins)
    fig4.add_trace(go.Bar(x=edges[:-1] + 50, y=counts / len(e) * 100, width=92, marker_color=color, showlegend=False,
                          hovertemplate='$%{x:,.0f}: %{y:.1f}% of workers<extra></extra>'), row=1, col=i)
    fig4.add_vline(x=NOMINAL_CUT, row=1, col=i, line_dash='dash', line_color='#fbbf24', line_width=2)
    xref = 'x domain' if i == 1 else f'x{i} domain'
    fig4.add_annotation(xref=xref, yref='paper', x=0.97, y=0.97, xanchor='right', yanchor='top', showarrow=False, align='right',
                        text=f"{(e >= NOMINAL_CUT).mean() * 100:.1f}% at or above the cut<br>{(e >= 6000).mean() * 100:.1f}% above $6,000 (off chart)",
                        font=dict(size=12, color='#e2e8f0'), bgcolor='rgba(11,18,32,0.85)', borderpad=4)
style_explore(fig4, "Where the cut falls in the distribution",
              "Weekly earnings in $100 bins. The tall 2023 bar is the topcode. Dashed line: the $2,884.61 cut",
              height=520)
fig4.update_xaxes(tickprefix='$', tickformat=',', range=[0, 6000], tickvals=[0, 2000, 4000])
fig4.update_yaxes(ticksuffix='%', title_text='% of workers', row=1, col=1)
fig4.show()

In [55]:
# Section 6 settings (robustness to the earnings cut)
TRIM_SHARE_POOLED = 0.12          # drop the top 12% of earners each month, genders pooled
CPI_BASE = SAMPLE_END             # month whose dollars the $150,000 cap is expressed in
CPI_SERIES = 'CUUR0000SA0'        # BLS CPI-U, all items, U.S. city average, not seasonally adjusted

In [56]:
# ---- CPI import: monthly CPI-U from the Bureau of Labor Statistics (BLS) -----------------------------------
# Used by the "CPI-deflated cap" check below. Downloaded from the BLS public API (version 2 accepts a start and end
# year without a registration key; up to 10 years per request), saved in the data folder, and downloaded again only
# if the sample needs months that the saved file lacks. Delete the file to force a fresh download.
CPI_FILE = DATA_DIR / 'cpi_u_nsa.csv'


def fetch_cpi(start_year, end_year, series=CPI_SERIES):
    """Monthly CPI-U (all items, not seasonally adjusted) from the BLS public API, as a dataframe (YEAR, MONTH, cpi)."""
    rows = []
    for first in range(start_year, end_year + 1, 10):                     # at most 10 years per request
        last = min(first + 9, end_year)
        resp = requests.get(f"https://api.bls.gov/publicAPI/v2/timeseries/data/{series}",
                            params={'startyear': first, 'endyear': last}, timeout=30)
        resp.raise_for_status()
        payload = resp.json()
        if payload.get('status') != 'REQUEST_SUCCEEDED':
            raise RuntimeError(f"BLS API did not return data: {payload.get('message')}")
        data = payload['Results']['series'][0]['data']
        missing_years = set(range(first, last + 1)) - {int(item['year']) for item in data}
        if missing_years:                                                  # the API can silently return only the latest years
            raise RuntimeError(f"BLS returned no data for {sorted(missing_years)} (asked for {first}-{last}).")
        for item in data:
            month = item['period']                                         # 'M01'...'M12'; 'M13' is the annual average
            if month.startswith('M') and month != 'M13' and item['value'] != '-':   # '-' = not collected (October 2025)
                rows.append({'YEAR': int(item['year']), 'MONTH': int(month[1:]), 'cpi': float(item['value'])})
    return pd.DataFrame(rows).drop_duplicates(['YEAR', 'MONTH']).sort_values(['YEAR', 'MONTH']).reset_index(drop=True)


needed_months = set(zip(wide['YEAR'], wide['MONTH'])) | {CPI_BASE}
cpi = pd.read_csv(CPI_FILE) if CPI_FILE.exists() else None
if cpi is None or needed_months - set(zip(cpi['YEAR'], cpi['MONTH'])):
    cpi = fetch_cpi(SAMPLE_START[0], max(year for year, _ in needed_months))
    cpi.to_csv(CPI_FILE, index=False)
    print(f"Downloaded CPI-U from BLS: {len(cpi)} months")

missing_cpi = sorted(needed_months - set(zip(cpi['YEAR'], cpi['MONTH'])))
assert not missing_cpi, (f"No CPI value for {len(missing_cpi)} months, from {tuple(map(int, missing_cpi[0]))} to "
                         f"{tuple(map(int, missing_cpi[-1]))}. BLS may not have published these months yet.")
base_cpi = cpi.loc[(cpi['YEAR'] == CPI_BASE[0]) & (cpi['MONTH'] == CPI_BASE[1]), 'cpi'].iloc[0]
print(f"CPI-U (1982-84 = 100): {len(cpi)} months, {cpi['YEAR'].min()} to {cpi['YEAR'].max()}; "
      f"base month {CPI_BASE[0]}-{CPI_BASE[1]:02d} = {base_cpi}")
print(f"The ${TOPCODE_WEEKLY:,.2f} weekly cap, in each year's dollars, is about "
      f"${TOPCODE_WEEKLY * cpi[cpi['YEAR'] == SAMPLE_START[0]]['cpi'].mean() / base_cpi:,.0f} in {SAMPLE_START[0]}")

CPI-U (1982-84 = 100): 200 months, 2010 to 2026; base month 2026-09 = 336.94
The $2,884.61 weekly cap, in each year's dollars, is about $1,867 in 2010


In [57]:
# ---- The alternative earnings cuts and the pipeline that applies them -------------------------------------------
def add_analysis_columns(df):
    """Section 3's variable construction, written as a function so the same steps can be applied to every version of the sample."""
    d = df.copy()
    d['co'] = d['STATEFIP'] == '08'
    d['ca'] = d['STATEFIP'] == '06'
    d['wa'] = d['STATEFIP'] == '53'
    group = np.select([d['ca'] == 1, d['co'] == 1, d['wa'] == 1], [1, 2, 3], default=0)
    d['comp_cat4'] = pd.Series(group, index=d.index).map(
        {0: 'Rest of US', 1: 'California', 2: 'Colorado', 3: 'Washington'}).astype('category')
    d['earnyear'] = d['EARNWEEK2'] * 52
    d['log_earnyear'] = np.log(d['earnyear'])
    d['epewa'] = (d['YEAR'] >= 2021).astype(int)
    d['epewa_own'] = (d['YEAR'] >= d['STATEFIP'].map(LAW_START_YEAR).fillna(2021)).astype(int)   # California and Washington: 2023
    d['female'] = (d['SEX'] == 2).astype(int)

    hisp = (d['HISPAN'] >= 100).astype(int)
    race_conditions = [hisp == 1,
                       (d['RACE'] == 100) & (hisp == 0),
                       d['RACE'].isin([200, 801]) & (hisp == 0),
                       d['RACE'].isin([300, 802]) & (hisp == 0),
                       d['RACE'].isin([651, 803]) & (hisp == 0),
                       (d['RACE'].isin([652]) | (d['RACE'] >= 804)) & (hisp == 0)]
    d['race_cat'] = np.select(race_conditions, ['Hispanic', 'White', 'Black', 'Indian', 'Asian', 'Other Race'], default='Not Specified')

    d['age'] = d['AGE'].astype('category').cat.rename_categories(lambda x: f"Age {x}")
    d['educ'] = d['EDUC'].map(educ_crosswalk).astype('category')
    d['ind_cat20'] = [lookup_category(y, c) for y, c in zip(d['YEAR'], d['IND'])]
    d['ind_cat20_label'] = d['ind_cat20']

    assert (d['race_cat'] != 'Not Specified').all(), "Some workers have an unrecognized RACE/HISPAN code."
    assert d['educ'].notna().all(), "Some workers have an EDUC code missing from educ_crosswalk."
    assert (d['ind_cat20'] != 'N/A').all(), "Some workers have no industry match."
    return d


def cut_cpi(df):
    """Keep workers below $150,000 a year expressed in constant dollars: the cap is TOPCODE_WEEKLY in the dollars of
    CPI_BASE, scaled to each worker's month with the CPI."""
    d = df.merge(cpi, on=['YEAR', 'MONTH'], how='left')
    assert d['cpi'].notna().all(), "Some workers' months have no CPI value."
    cap = TOPCODE_WEEKLY * d['cpi'] / base_cpi
    # Before the topcode became dynamic (April 2024) censored earnings sit exactly at TOPCODE_WEEKLY, so the cap
    # must not exceed it in those months or censored values would stay in the sample.
    before_dynamic = (d['YEAR'] * 100 + d['MONTH']) < 202404
    assert (cap[before_dynamic] <= TOPCODE_WEEKLY + 1e-9).all(), "CPI_BASE is too early: the cap exceeds the topcode."
    return d[d['EARNWEEK2'] < cap].drop(columns='cpi')


def trim_top_share(df, share, by):
    """Drop everyone at or above the (1 - share) quantile of weekly earnings within each group in `by`
    (ties at the threshold, such as rounded values or the topcode, are dropped too)."""
    threshold = df.groupby(by)['EARNWEEK2'].transform(lambda s: s.quantile(1 - share))
    return df[df['EARNWEEK2'] < threshold]


def largest_monthly_male_share_above_cut(df):
    """The largest share of men, in any single month, that the nominal cut removes."""
    above = df['EARNWEEK2'] >= TOPCODE_WEEKLY
    share = above.groupby([df['YEAR'], df['MONTH'], df['SEX']]).mean().unstack()
    return share[1].max()                                                  # SEX 1 = men


ROBUSTNESS_CUTS = {                     # short label, long label
    'nominal': ('Nominal cut', f"Nominal cut (${TOPCODE_WEEKLY:,.2f} a week)"),
    'cpi': ('CPI-deflated cap', "$150,000 cap in constant dollars (CPI-deflated)"),
    'top_pooled': (f"Top {TRIM_SHARE_POOLED:.0%}", f"Top {TRIM_SHARE_POOLED:.0%} of earners each month (genders pooled)"),
    'top_by_gender': ('Within-gender trim', "Top X% within each gender (X = largest monthly share of men the nominal cut removes)"),
}


def build_variant(method):
    """The analysis sample under one earnings cut, with the minimum wage exclusion applied first so that trimmed
    shares refer to workers above the minimum wage."""
    d = wide.copy()
    d['STATEFIP'] = d['STATEFIP'].astype(str).str.pad(width=2, fillchar='0')
    d = apply_min_wage_exclusion(d, consol_mw[['State'] + year_columns])
    info = {'above minimum wage': len(d)}
    if method == 'nominal':
        d = d[d['EARNWEEK2'] < TOPCODE_WEEKLY]
    elif method == 'cpi':
        d = cut_cpi(d)
    elif method == 'top_pooled':
        d = trim_top_share(d, TRIM_SHARE_POOLED, ['YEAR', 'MONTH'])
    elif method == 'top_by_gender':
        x = np.ceil(largest_monthly_male_share_above_cut(d) * 1000) / 1000   # rounded up to 0.1%
        info['trim share'] = x
        d = trim_top_share(d, x, ['YEAR', 'MONTH', 'SEX'])
    else:
        raise ValueError(method)
    sample = add_analysis_columns(d.reset_index(drop=True))
    info['kept'] = len(sample)
    info['removed %'] = 100 * (1 - len(sample) / info['above minimum wage'])
    return sample, info

In [58]:
# ---- Placebo test: treat each state in turn as if it were the "treated" state -----------------------------------
# Triple-difference regression on a pool that excludes California and Washington (their laws are the other treatments):
#   log earnings ~ female * post * treated  +  controls
# where `treated` marks one state at a time. The coefficient on female x post x treated is how much more that state's
# gender gap narrowed after 2021 than in the rest of the pool. Colorado's coefficient is then ranked among all states.
import patsy

PLACEBO_CONTROLS = 'C(age) + C(race_cat) + C(educ)'          # the industry specification adds + C(ind_cat20_label)


def placebo_coefficients(sample, controls=PLACEBO_CONTROLS):
    """Triple-difference coefficient (female x post x treated) for each state in the pool, as a Series by state name.
    Uses the common January 2021 date (`epewa`) for every state. The controls enter once, as in the main regressions."""
    pool = sample[sample['comp_cat4'].isin(['Colorado', 'Rest of US'])].reset_index(drop=True)
    X = np.asarray(patsy.dmatrix(controls, pool, return_type='matrix'))        # includes the intercept
    y = pool['log_earnyear'].to_numpy()
    f = pool['female'].to_numpy(dtype=float)
    p = pool['epewa'].to_numpy(dtype=float)
    XtX, Xty = X.T @ X, X.T @ y
    coefs = {}
    for code in sorted(pool['STATEFIP'].unique()):
        t = (pool['STATEFIP'] == code).to_numpy(dtype=float)
        Z = np.column_stack([f, p, t, f * p, f * t, p * t, f * p * t])
        XtZ = X.T @ Z
        A = np.block([[XtX, XtZ], [XtZ.T, Z.T @ Z]])
        b = np.concatenate([Xty, Z.T @ y])
        coefs[fips_to_state.get(code, code)] = np.linalg.lstsq(A, b, rcond=None)[0][-1]      # last term: f x p x t
    return pd.Series(coefs).sort_values(ascending=False)

In [59]:
# ---- Run the main model under each cut ------------------------------------------------------------------------
# This fits the model four times in two specifications; expect a few minutes.
robust = {}
for method, (short_label, long_label) in ROBUSTNESS_CUTS.items():
    sample, info = build_variant(method)
    _, no_ind = run_by_group(sample, 'comp_cat4', FORMULA_NO_IND, post_col='epewa_own')
    _, with_ind = run_by_group(sample, 'comp_cat4', FORMULA_IND, post_col='epewa_own')
    robust[method] = {'no_industry': no_ind, 'industry': with_ind, **info}
    robust[method]['placebo'] = {'no_industry': placebo_coefficients(sample, PLACEBO_CONTROLS),
                            'industry': placebo_coefficients(sample, PLACEBO_CONTROLS + ' + C(ind_cat20_label)')}
    print(f"{long_label}: kept {info['kept']:,} workers ({info['removed %']:.1f}% of those above the minimum wage)"
          + (f"; trim share {info['trim share']:.1%}" if 'trim share' in info else ""))

# The nominal cut must reproduce the main analysis exactly.
_, main_check = run_by_group(ipums_df, 'comp_cat4', FORMULA_NO_IND, post_col='epewa_own')
assert robust['nominal']['kept'] == len(ipums_df), "The nominal variant does not have the same sample as the main analysis."
assert np.allclose(robust['nominal']['no_industry'].sort_index()['interaction_coef'], main_check.sort_index()['interaction_coef']), \
    "The nominal variant does not reproduce the main regression."
print("Check passed: the nominal variant reproduces the main analysis.")

GROUPS = ['Colorado', 'Washington', 'California', 'Rest of US']


def interaction_table(spec):
    """Change in women's earnings as a share of men's after the law, in cents on the dollar (exact ratio change)."""
    rows = {}
    for method, (short_label, _) in ROBUSTNESS_CUTS.items():
        coef = robust[method][spec]['ratio_change'] * 100
        rows[short_label] = {**{g: coef[g] for g in GROUPS}, 'Colorado minus Rest of US': coef['Colorado'] - coef['Rest of US']}
    return pd.DataFrame(rows).T.round(1)


print("\nChange in women's earnings as cents on the dollar of men's after the law, no industry controls")
display(interaction_table('no_industry'))
print("With industry controls")
display(interaction_table('industry'))
print("Workers kept under each cut")
display(pd.DataFrame({ROBUSTNESS_CUTS[m][0]: {'kept': robust[m]['kept'], 'removed % of workers above the minimum wage': round(robust[m]['removed %'], 1)}
                      for m in ROBUSTNESS_CUTS}).T)


def plot_variants(table, headline, controls):
    """Grouped bars: one cluster per version of the cut, one bar per comparison group."""
    fig = go.Figure()
    for group in GROUPS:
        fig.add_trace(go.Bar(name=group, x=list(table.index), y=table[group], marker_color=GROUP_COLORS[group],
                             text=[f"{v:.1f}" for v in table[group]], textposition='outside', textfont=dict(color='#f1f5f9', size=12),
                             hovertemplate='%{x}, ' + group + ': %{y:.1f} <extra></extra>'))
    style_explore(fig, headline, f"Change in women's earnings relative to men's after the law, in cents on the dollar. Controls: {controls}", height=560)
    fig.update_layout(barmode='group', bargap=0.25, bargroupgap=0.06)
    fig.update_yaxes(ticksuffix='', title_text='Cents on the dollar')
    fig.show()
    return fig


fig = plot_variants(interaction_table('no_industry'), "The result under four earnings cuts", "age, education and race")
fig = plot_variants(interaction_table('industry'), "The result under four earnings cuts", "age, education, race and industry")

Nominal cut ($2,884.61 a week): kept 588,770 workers (5.6% of those above the minimum wage)
$150,000 cap in constant dollars (CPI-deflated): kept 558,116 workers (10.5% of those above the minimum wage)
Top 12% of earners each month (genders pooled): kept 546,713 workers (12.4% of those above the minimum wage)
Top X% within each gender (X = largest monthly share of men the nominal cut removes): kept 529,598 workers (15.1% of those above the minimum wage); trim share 14.7%
Check passed: the nominal variant reproduces the main analysis.

Change in women's earnings as cents on the dollar of men's after the law, no industry controls


,Colorado,Washington,California,Rest of US,Colorado minus Rest of US
Nominal cut,7.5000,4.6000,2.1000,1.5000,6.1000
CPI-deflated cap,5.5000,2.7000,1.2000,0.4000,5.1000
Top 12%,4.5000,2.0000,0.7000,-0.1000,4.6000
Within-gender trim,4.7000,2.3000,1.9000,0.8000,3.9000


With industry controls


,Colorado,Washington,California,Rest of US,Colorado minus Rest of US
Nominal cut,7.3000,4.4000,2.1000,1.5000,5.8000
CPI-deflated cap,5.2000,2.3000,1.4000,0.5000,4.7000
Top 12%,4.3000,1.6000,0.9000,0.0000,4.3000
Within-gender trim,4.4000,1.9000,2.0000,0.9000,3.5000


Workers kept under each cut


,kept,removed % of workers above the minimum wage
Nominal cut,588770.0000,5.6000
CPI-deflated cap,558116.0000,10.5000
Top 12%,546713.0000,12.4000
Within-gender trim,529598.0000,15.1000


In [60]:
from scipy import stats

def difference_test(spec):
    rows = {}
    for m, (short, _) in ROBUSTNESS_CUTS.items():
        g = robust[m][spec]
        diff = g.loc['Colorado', 'interaction_coef'] - g.loc['Rest of US', 'interaction_coef']
        se = np.hypot(g.loc['Colorado', 'interaction_se'], g.loc['Rest of US', 'interaction_se'])
        rows[short] = {'Colorado minus Rest of US (points)': diff * 100, 'std. error': se * 100, 'z': diff / se,
                        'p-value': 2 * (1 - stats.norm.cdf(abs(diff / se)))}
    return pd.DataFrame(rows).T.round(3)

for spec in ['no_industry', 'industry']:
    print(spec)
    display(difference_test(spec))

no_industry


,Colorado minus Rest of US (points),std. error,z,p-value
Nominal cut,7.1770,1.9680,3.6470,0.0000
CPI-deflated cap,5.8570,1.9050,3.0750,0.0020
Top 12%,5.2860,1.8930,2.7930,0.0050
Within-gender trim,4.6350,1.8500,2.5050,0.0120


industry


,Colorado minus Rest of US (points),std. error,z,p-value
Nominal cut,6.5460,1.9140,3.4200,0.0010
CPI-deflated cap,5.2670,1.8570,2.8370,0.0050
Top 12%,4.7800,1.8440,2.5920,0.0100
Within-gender trim,4.0840,1.8050,2.2630,0.0240


## Placebo test: is Colorado unusual?

The comparison tables above show that the estimated Colorado effect depends on the earnings cut. They do not show whether an effect of that size could appear by chance in any state with this sample size, especially after the data are truncated on earnings. The placebo test asks that question directly.

For each of the 49 states other than California and Washington, the test pretends that state passed a law in January 2021 and estimates the same triple-difference for it: female × post-2021 × that state, against a pool of the remaining states. The pool excludes California and Washington, which had their own pay transparency laws. Doing this for every state gives a distribution of effects for states with no relevant law. Colorado's real estimate is then compared with that distribution. The tables report Colorado's effect, its rank among the 49 states, the strongest other state, the median state, and the share of states with an effect at or above Colorado's (a rank-based p-value). Effects here are in log points × 100, so they differ from the cents-on-the-dollar figures above.

**Placebo Test Results.** With age, education and race controls, Colorado ranks first under three of the four cuts (6.8 under the main nominal cut, 5.4 under the CPI-deflated cap, 4.2 under the within-gender trim) and second under the top 12% trim (4.8). The share of states at or above Colorado is 2.0% under the three first-place cuts and 4.1% under the top 12% trim. With industry controls Colorado ranks first under the nominal cut (6.3), second under the CPI-deflated cap (4.9), and third under both the top 12% trim (4.4) and the within-gender trim (3.7), with 2.0% to 6.1% of states at or above it. The median placebo state is close to zero in every row (0.0 to 0.5 in the first table, 0.1 to 0.3 in the second), so the typical state with no law shows no effect. South Dakota is the strongest other state in every row, with effects of 4.1 to 5.4, which is large enough to be worth a look at its sample size.

**Interpretation.** Colorado is in the top three of 49 states under every cut and specification, and well above the typical state. That supports the main result: the Colorado effect is not simply what truncation on earnings produces anywhere. It also shows the limits of the evidence. Under the stricter cuts with industry controls, Colorado is no longer the clear top state, and the rank-based p-values reach 6.1%, just above the usual 5% threshold. The placebo test supports a real Colorado effect that is smaller than the main nominal estimate, not an effect that is statistically ironclad. With only about 50 observations per month in Colorado, that is the level of confidence these data can support.

In [61]:
# ---- Placebo results ---------------------------------------------------------------------------------------------
def placebo_summary(spec):
    """Colorado's placebo coefficient (log points x 100), its rank among all states, and the closest competitor."""
    rows = {}
    for method, (short_label, _) in ROBUSTNESS_CUTS.items():
        c = robust[method]['placebo'][spec] * 100
        ranked = c.sort_values(ascending=False)
        others = ranked.drop('Colorado')
        rows[short_label] = {'Colorado': round(c['Colorado'], 1),
                             'Colorado rank': f"{ranked.index.get_loc('Colorado') + 1} of {len(ranked)}",
                             'next best state': f"{others.index[0]} ({others.iloc[0]:.1f})",
                             'median state': round(c.median(), 1),
                             'share of states at or above Colorado': round((c >= c['Colorado']).mean(), 3)}
    return pd.DataFrame(rows).T


def plot_placebo(spec, headline, controls):
    """One row per version of the cut: each grey dot is a state's placebo coefficient, the green dot is Colorado."""
    jitter = np.random.default_rng(0)
    fig = go.Figure()
    for i, (method, (short_label, _)) in enumerate(ROBUSTNESS_CUTS.items()):
        c = robust[method]['placebo'][spec] * 100
        others = c.drop('Colorado')
        fig.add_trace(go.Scatter(x=others.values, y=i + jitter.uniform(-0.2, 0.2, len(others)), mode='markers',
                                 marker=dict(color='#94a3b8', size=9, opacity=0.75), text=others.index, showlegend=False,
                                 hovertemplate='%{text}: %{x:.1f}<extra></extra>'))
        fig.add_trace(go.Scatter(x=[c['Colorado']], y=[i], mode='markers+text', text=[f"Colorado {c['Colorado']:.1f}"],
                                 textposition='top center', textfont=dict(color='#34d399', size=13), showlegend=False,
                                 marker=dict(color='#34d399', size=18, line=dict(color='#f8fafc', width=2)),
                                 hovertemplate='Colorado: %{x:.1f}<extra></extra>'))
    style_explore(fig, headline, f"Each dot treats one state as if it passed the law in 2021. Controls: {controls}", height=520)
    fig.update_yaxes(tickmode='array', tickvals=list(range(len(ROBUSTNESS_CUTS))),
                     ticktext=[v[0] for v in ROBUSTNESS_CUTS.values()], autorange='reversed')
    fig.update_xaxes(title_text="Extra narrowing of the gender gap relative to the other states (log points x 100)",
                     zeroline=True, zerolinecolor='rgba(148,163,184,0.5)')
    fig.show()
    return fig


for spec, controls in [('no_industry', 'age, education and race'), ('industry', 'age, education, race and industry')]:
    print(f"Placebo test, {controls}")
    display(placebo_summary(spec))
    fig = plot_placebo(spec, "Colorado against every other state", controls)

Placebo test, age, education and race


,Colorado,Colorado rank,next best state,median state,share of states at or above Colorado
Nominal cut,6.8000,1 of 49,South Dakota (4.2),0.5000,0.0200
CPI-deflated cap,5.4000,1 of 49,South Dakota (4.8),0.0000,0.0200
Top 12%,4.8000,2 of 49,South Dakota (5.1),0.2000,0.0410
Within-gender trim,4.2000,1 of 49,South Dakota (4.1),0.5000,0.0200


Placebo test, age, education, race and industry


,Colorado,Colorado rank,next best state,median state,share of states at or above Colorado
Nominal cut,6.3000,1 of 49,South Dakota (4.8),0.3000,0.0200
CPI-deflated cap,4.9000,2 of 49,South Dakota (5.3),0.3000,0.0410
Top 12%,4.4000,3 of 49,South Dakota (5.4),0.1000,0.0610
Within-gender trim,3.7000,3 of 49,South Dakota (4.5),0.2000,0.0610


# Section 7: Supplementary Analysis

This section of code seeks to analyze some supplementary considerations. These include:

1. Is the reduction in the gender wage gap in Colorado being driven by women's wages going up, or men's wages going down? This is accomplished by repeating the primary regression but with men only in the sample. This compares Colorado men's wages to the rest of the men in the United States.
2. Is the reduction within Colorado determined by a specific industry? This was evaluated by splitting the Colorado subsample into industry-specific samples and running the regression without industry controls. The analysis runs into significant issues due to small sample size. An industry is reported only if each of its four gender-by-period groups has at least 10 observations and the regression returns valid standard errors; the rest are listed as not reported.
3. Is the reduction in the gender wage gap in Colorado driven by urban or rural parts of the state? Urban Colorado is defined as the Denver, Colorado Springs, Fort Collins, Greeley, Pueblo and Boulder metropolitan areas (from the IPUMS METFIPS variable). Again, this runs into significant issues due to small sample size.
4. What is the increase in women's wages relative to men's in terms of dollars (not percentage points)? These are nominal dollars as reported in each year; no inflation adjustment is made.

In [62]:
FORMULA_MEN = 'log_earnyear ~ C(epewa) + C(co) + C(epewa):C(co) + C(age) + C(ind_cat20_label) + C(race_cat) + C(educ)'

male_subset = ipums_df[ipums_df['female'] == 0]
model_men = smf.ols(FORMULA_MEN, data=male_subset).fit(cov_type='HC3')

print('Regression Measuring Change in Colorado Male Wages Relative to US, Before and After EPEWA')
print('Observations:', int(model_men.nobs))
display(term_table(model_men, ['epewa', 'C(co)']))

Regression Measuring Change in Colorado Male Wages Relative to US, Before and After EPEWA
Observations: 322910


,coef,std_err,p_value
C(epewa)[T.1],0.2340,0.0016,0.0000
C(co)[T.True],0.0395,0.0064,0.0000
C(epewa)[T.1]:C(co)[T.True],0.0254,0.0131,0.0530


There is no evidence of a drop in Colorado men's wages relative to men in the rest of the United States. Colorado men's wages rose by about 2.5 percent more than men's elsewhere after EPEWA (log-point coefficient 0.025, p = 0.053), which is significant at the 10% level but not at the 5% level. In the July 2024 analysis the figure was about 1.6 percent (p = 0.32). Some of the narrowing of the gap in Colorado may therefore reflect broader wage growth in the state.

In [63]:
FORMULA_CO_IND = 'log_earnyear ~ C(epewa) + C(female) + C(female):C(epewa) + C(age) + C(race_cat) + C(educ)'

co_df = ipums_df[ipums_df['co'] == 1]
rows = []
for industry, subset in co_df.groupby('ind_cat20_label', observed=True):
    cells = subset.groupby(['female', 'epewa']).size()      # observations in each gender x period cell
    row = {'Industry Label': industry, 'n': len(subset), 'min_cell': cells.min() if len(cells) == 4 else 0}
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')   # degenerate fits are screened out below (group sizes, p-values)
            model, key = fit_did(subset, FORMULA_CO_IND)
            row.update({'max_leverage': model.get_influence().hat_matrix_diag.max(), **key})
    except Exception as e:                                   # e.g. a singular design matrix
        row.update({'max_leverage': np.nan, 'error': str(e)[:60]})
    rows.append(row)

co_industry_diag = pd.DataFrame(rows).set_index('Industry Label').sort_values('max_leverage', ascending=False)
display(co_industry_diag.round(4))

,n,min_cell,max_leverage,female_coef,female_pval,interaction_coef,interaction_pval,interaction_se,error
Industry Label,,,,,,,,,
"Arts, Entertainment, and Recreation:",173,16,1.0000,-0.1201,1.0000,0.1343,1.0000,inf,NaN
Real Estate and Rental and Leasing:,211,24,1.0000,-0.0879,0.2932,0.1080,0.5531,0.1821,NaN
Administrative and Support and Waste Management and Remediation Services:,357,24,1.0000,-0.0949,0.1301,-0.0711,0.6031,0.1367,NaN
Public Administration:,608,63,1.0000,-0.1419,0.0002,-0.1006,0.1814,0.0752,NaN
"Agriculture, Forestry, Fishing and Hunting:",107,10,1.0000,-0.4448,0.0236,0.1801,0.5813,0.3265,NaN
Other Services (Except Public Administration):,374,29,1.0000,-0.2566,0.0000,0.1447,0.2273,0.1198,NaN
Health Care and Social Assistance:,1258,67,1.0000,-0.1280,0.0001,0.0683,0.2940,0.0651,NaN
"Mining, Quarrying, and Oil and Gas Extraction:",140,6,1.0000,-0.3003,0.0816,0.1185,0.6743,0.2821,NaN
Wholesale Trade:,267,14,1.0000,-0.1608,0.0289,-0.1362,0.4565,0.1829,NaN


In [64]:
MIN_CELL = 10   # minimum observations in each gender x period group for an industry to be reported

# A fit is unusable if the HC3 standard errors degenerate: missing or exactly-1.0 p-values
pvals = co_industry_diag[['female_pval', 'interaction_pval']]
enough_obs = co_industry_diag['min_cell'] >= MIN_CELL
valid_fit = pvals.notna().all(axis=1) & (pvals < 0.9999).all(axis=1)
usable = enough_obs & valid_fit

print(f"Reported industries: {usable.sum()} of {len(co_industry_diag)}")
print("Not reported (fewer than %d in a gender x period group):" % MIN_CELL, ", ".join(co_industry_diag.index[~enough_obs]) or "none")
print("Not reported (degenerate fit):", ", ".join(co_industry_diag.index[enough_obs & ~valid_fit]) or "none")

co_industry_results = co_industry_diag[usable].copy()
co_industry_results['Pre-EPEWA Wage Gap'] = np.exp(co_industry_results['female_coef'])
co_industry_results['Post-EPEWA Wage Gap'] = np.exp(co_industry_results['female_coef'] + co_industry_results['interaction_coef'])

# Asterisk where the interaction p-value is above 0.10
co_industry_results.index = [f"{name}*" if p > 0.10 else name
                             for name, p in zip(co_industry_results.index, co_industry_results['interaction_pval'])]
co_industry_results = co_industry_results.sort_values('interaction_pval')

display(co_industry_results[['n', 'min_cell', 'female_coef', 'female_pval', 'interaction_coef', 'interaction_pval',
                             'Pre-EPEWA Wage Gap', 'Post-EPEWA Wage Gap']].round(4))

Reported industries: 15 of 20
Not reported (fewer than 10 in a gender x period group): Mining, Quarrying, and Oil and Gas Extraction:, Utilities:, Management of Companies and Enterprises:
Not reported (degenerate fit): Arts, Entertainment, and Recreation:, Transportation and Warehousing:


,n,min_cell,female_coef,female_pval,interaction_coef,interaction_pval,Pre-EPEWA Wage Gap,Post-EPEWA Wage Gap
"Professional, Scientific, and Technical Services:",1013,115,-0.3010,0.0000,0.2701,0.0000,0.7401,0.9695
Manufacturing:,826,39,-0.2136,0.0000,0.1925,0.0216,0.8076,0.9791
Retail Trade:,874,79,-0.1876,0.0000,0.1520,0.0316,0.8290,0.9651
Educational Services:*,991,63,-0.1165,0.0007,0.1003,0.1486,0.8900,0.9839
Public Administration:*,608,63,-0.1419,0.0002,-0.1006,0.1814,0.8677,0.7847
Other Services (Except Public Administration):*,374,29,-0.2566,0.0000,0.1447,0.2273,0.7737,0.8941
Health Care and Social Assistance:*,1258,67,-0.1280,0.0001,0.0683,0.2940,0.8799,0.9421
Wholesale Trade:*,267,14,-0.1608,0.0289,-0.1362,0.4565,0.8514,0.7430
Accommodation and Food Services:*,484,26,-0.0670,0.1691,0.0771,0.5186,0.9352,1.0101
Real Estate and Rental and Leasing:*,211,24,-0.0879,0.2932,0.1080,0.5531,0.9158,1.0203


Three industries show a statistically significant increase in women's relative earnings at the 5% level: Professional, Scientific, and Technical Services (women's earnings rising from 74.0 to 97.0 cents on the dollar of men's, +22.9 cents), Manufacturing (80.8 to 97.9 cents, +17.1, p = 0.022) and Retail Trade (82.9 to 96.5 cents, +13.6, p = 0.032). Manufacturing was not significant in the July 2024 analysis. Mining, Utilities and Management of Companies are not reported because a gender-by-period group has fewer than 10 observations, and Arts, Entertainment, and Recreation and Transportation and Warehousing are not reported because their regressions return degenerate standard errors.
These three industries seem to be driving the change in the gender wage gap, though other industries may also be contributing. It is difficult to tell to what magnitude other industries are contributing.

Which industries are reported depends partly on numerical details of the robust standard errors in very small samples, so these industry results should be read as indicative.

In [65]:
# According to the Colorado demographer, the 'Front Range' urban corridor ranges from 70-90% of the state
# https://storymaps.arcgis.com/stories/f622f3746d7649749b103e1e2b018a35
# Metropolitan areas used as "urban" Colorado (METFIPS codes, 2014 onward):
# https://cps.ipums.org/cps/codes/metfips_2014onward_codes.shtml
URBAN_CO_METROS = {19740: 'Denver-Aurora-Lakewood', 
                   17820: 'Colorado Springs', 
                   22660: 'Fort Collins',
                   24540: 'Greeley', 
                   39380: 'Pueblo', 
                   14500: 'Boulder'}

ipums_df['urban_co'] = ((ipums_df['METFIPS'].isin(URBAN_CO_METROS)) & (ipums_df['co'] == 1)).astype(int)
ipums_df['rural_co'] = ((ipums_df['urban_co'] == 0) & (ipums_df['co'] == 1)).astype(int)

In [66]:
# Because the samples are so small, controls were dropped to take the simplest measure possible
FORMULA_SIMPLE = 'log_earnyear ~ C(epewa) + C(female) + C(epewa):C(female)'

for label, flag in [('Urban', 'urban_co'), ('Rural', 'rural_co')]:
    subset = ipums_df[ipums_df[flag] == 1]
    model = smf.ols(FORMULA_SIMPLE, data=subset).fit(cov_type='HC3')
    print(f'Simple Gender Wage Gap Model Results for {label} Colorado, No Controls (observations: {int(model.nobs):,})')
    display(term_table(model, ['epewa', 'female']))

Simple Gender Wage Gap Model Results for Urban Colorado, No Controls (observations: 8,713)


,coef,std_err,p_value
C(epewa)[T.1],0.2826,0.0160,0.0000
C(female)[T.1],-0.1609,0.0122,0.0000
C(epewa)[T.1]:C(female)[T.1],0.1221,0.0231,0.0000


Simple Gender Wage Gap Model Results for Rural Colorado, No Controls (observations: 1,232)


,coef,std_err,p_value
C(epewa)[T.1],0.2476,0.0394,0.0000
C(female)[T.1],-0.2287,0.0313,0.0000
C(epewa)[T.1]:C(female)[T.1],0.1108,0.0585,0.0581


With no controls, women's earnings in urban Colorado rose from 85.1 to 96.2 cents on the dollar of men's (+11.1 cents; log-point interaction 0.122, p < 0.001). In rural Colorado they rose from 79.6 to 88.9 cents (+9.3 cents; interaction 0.111, p = 0.058, significant at the 10% level but not the 5% level). The magnitudes are similar, and there does not appear to be a large difference between the rural and urban reduction in the gender wage gap using the simplest measure.

In [67]:
# Dollar value of the primary finding, in nominal dollars per year: industry controls
FORMULA_IND_DOLLARS = FORMULA_IND.replace('log_earnyear', 'earnyear')
models_dollars_ind, dollars_ind = run_by_group(ipums_df, 'comp_cat4', FORMULA_IND_DOLLARS, add_gaps=False, post_col='epewa_own')

print('Dollar value of the change in the gender gap, industry controls ($ per year)')
display(dollars_ind.round({'female_coef': 0, 'interaction_coef': 0, 'female_pval': 4, 'interaction_pval': 4}))

Dollar value of the change in the gender gap, industry controls ($ per year)


,female_coef,female_pval,interaction_coef,interaction_pval,interaction_se
California,-7923.0000,0.0000,744.0000,0.2535,651.1727
Colorado,-10185.0000,0.0000,3935.0000,0.0027,1310.0755
Rest of US,-9496.0000,0.0000,-472.0000,0.0026,156.7048
Washington,-9968.0000,0.0000,1401.0000,0.3324,1445.9141


The reduction in the gender wage gap with industry controls in Colorado is approximately $3,935 per year (p = 0.003), in nominal dollars as reported in each year. The dollar changes for Washington ($1,401, p = 0.33) and California ($744, p = 0.25) are not statistically significant under their January 2023 start dates.

In [68]:
# Dollar value of the primary finding, in nominal dollars per year: no industry controls
FORMULA_NO_IND_DOLLARS = FORMULA_NO_IND.replace('log_earnyear', 'earnyear')
models_dollars_no_ind, dollars_no_ind = run_by_group(ipums_df, 'comp_cat4', FORMULA_NO_IND_DOLLARS, add_gaps=False, post_col='epewa_own')

print('Dollar value of the change in the gender gap, no industry controls ($ per year)')
display(dollars_no_ind.round({'female_coef': 0, 'interaction_coef': 0, 'female_pval': 4, 'interaction_pval': 4}))

Dollar value of the change in the gender gap, no industry controls ($ per year)


,female_coef,female_pval,interaction_coef,interaction_pval,interaction_se
California,-8724.0000,0.0000,799.0000,0.2261,660.0609
Colorado,-12612.0000,0.0000,4264.0000,0.0015,1346.6456
Rest of US,-11291.0000,0.0000,-461.0000,0.0038,159.6351
Washington,-12606.0000,0.0000,1678.0000,0.2534,1468.8266


When removing industry controls, the reduction grows to approximately $4,264 per year (p = 0.002). In nominal dollars the gap in the rest of the country widened by about $460 over the same period, mostly reflecting nominal wage growth, so Colorado's figure is measured against a rising baseline. Washington ($1,678, p = 0.25) and California ($799, p = 0.23) are not statistically significant. The dollar figures use the main nominal earnings cut and are not repeated for the alternative cuts in Section 6.

The following cell saves results.

In [69]:
# ---- Save the key results for this sample window, so runs can be compared later --------------------------
RESULTS_DIR = DATA_DIR / 'results' / f"{SAMPLE_START[0]}-{SAMPLE_START[1]:02d}_{SAMPLE_END[0]}-{SAMPLE_END[1]:02d}"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

_, group_no_ind = run_by_group(ipums_df, 'comp_cat4', FORMULA_NO_IND, post_col='epewa_own')
_, group_ind = run_by_group(ipums_df, 'comp_cat4', FORMULA_IND, post_col='epewa_own')

tables = {
    'groups_no_industry': group_no_ind,
    'groups_industry': group_ind,
    'states_no_industry': ranking_no_ind,
    'states_industry': ranking_ind,
    'colorado_dollars_no_industry': dollars_no_ind,
    'colorado_dollars_industry': dollars_ind,
    'colorado_by_industry': co_industry_results,
    'colorado_men_only': term_table(model_men, ['epewa', 'C(co)']),
}

# Section 6: robustness tables (cents on the dollar), the difference test and the placebo summary
for spec in ['no_industry', 'industry']:
    tables[f'robustness_cents_{spec}'] = interaction_table(spec)
    tables[f'robustness_difference_test_{spec}'] = difference_test(spec)
    tables[f'robustness_placebo_{spec}'] = placebo_summary(spec)

for name, table in tables.items():
    table.to_csv(RESULTS_DIR / f"{name}.csv")

run_info = pd.Series({
    'sample_start': f"{SAMPLE_START[0]}-{SAMPLE_START[1]:02d}",
    'sample_end': f"{SAMPLE_END[0]}-{SAMPLE_END[1]:02d}",
    'monthly_samples': len(CpsSamples),
    'rows_after_exclusion': len(ipums_df),
    'run_date': pd.Timestamp.now().strftime('%Y-%m-%d %H:%M'),
})
run_info.to_csv(RESULTS_DIR / 'run_info.csv', header=False)
print(f"Saved {len(tables)} tables to {RESULTS_DIR.relative_to(DATA_DIR)}")
print(run_info)

Saved 14 tables to results\2010-01_2026-09
sample_start                     2010-01
sample_end                       2026-09
monthly_samples                      200
rows_after_exclusion              588770
run_date                2026-10-28 16:51
dtype: object


# Conclusion

The key findings from this analysis, using monthly CPS data from January 2010 through September 2026, are:

> Colorado has reduced the gender wage gap faster than any other state since implementing EPEWA. Washington shows a similar but smaller result, and California a smaller one.
> The reduction in Colorado is equivalent to women being paid approximately $3,935 more per year with industry controls and $4,264 more without, measured in nominal dollars as reported in each year.

This was determined using a difference in difference experimental design. These results were consistent when controlling for industry and not. This analysis utilizes the middle 60% of the U.S. income distribution. Under the main earnings cut, women's earnings in Colorado rose by 7.5 cents on the dollar of men's earnings without industry controls and 7.3 with, compared with 4.6 and 4.4 for Washington, 2.1 for California (both specifications), and 1.5 for the rest of the country. California and Washington are measured from January 2023, when their own laws took effect.

> The Colorado result is positive under every earnings cut tested, but its size depends on the cut.

The main analysis drops workers earning $2,884.61 a week or more, a nominal cut that removes a growing share of workers over time (about 3% in 2010 and 11% in 2026) and falls more heavily on high-wage states. Section 6 tests alternatives: a $150,000 cap deflated by the CPI, a trim of the top 12% of earners each month, and a within-gender trim. Under these the Colorado effect is 4.3 to 5.5 cents, compared with 7.3 to 7.5 under the main cut, and the difference between Colorado and the rest of the country is 3.5 to 6.1 cents and statistically significant in every case. A placebo test that treats each of 49 other states as the treated state ranks Colorado first, second or third in every case, with no more than 6.1% of states at or above it. The estimate of 7.5 cents should therefore be read as an upper end of a range, with 4 to 5 cents a more conservative figure once the earnings cut is addressed.

The dollar figures are not adjusted for inflation, so they should be read as a lower bound in present-value terms. Part of the increase over the July 2024 figures ($3,304 and $3,358) reflects price growth, because the nominal dollar gap in the rest of the country also widened over the longer window. The percentage results are less affected.

The reduction in the gender wage gap in Colorado is larger when the industry control is not applied. This implies women are transitioning to higher paying industries after EPEWA came into effect.

> Colorado went from a near-median gender wage gap among the 50 states and DC (24th of 51 before the law without industry controls, 22nd with them) to ranking 3rd after EPEWA without industry controls, behind DC and California, and 2nd with them, behind DC. It had the largest reduction in the gender wage gap among the 50 states and DC in both specifications.

At the state level, 20 of 51 jurisdictions show a statistically significant change at the 5% level in both specifications (up from 14 and 16 in the July 2024 analysis). Not all state-level findings are significant, but their cardinality can provide some insight.

> The gender wage gap in Colorado is closing mainly because women's wages are rising relative to men's, not because men's wages are falling.

There is no evidence that Colorado men's wages are falling behind the nation. Colorado men's wages rose about 2.5 percent more than men's elsewhere after EPEWA (p = 0.053), so some of the narrowing may reflect broader wage growth in Colorado.

> The reduction in Colorado is concentrated in Professional, Scientific, and Technical Services, Manufacturing and Retail Trade. In Professional Services, women now earn 97 cents for every dollar a man earns.

The rest of the industry samples are too small to make meaningful conclusions. Colorado has about 50 observations per month across all workers, and fewer within industries.

> The trend is consistent across urban and rural Colorado.

Due to sample size restrictions, this finding took the simplest measure of the gender wage gap and found that, with no controls, women's earnings relative to men's rose by a similar amount in urban (11.1 cents on the dollar) and rural (9.3 cents, significant at the 10% level but not the 5% level) Colorado after EPEWA came into effect.

### Results compared with the July 2024 analysis

The July 2024 figures were reported as the interaction coefficient (log points x 100), so that is the unit compared in the first rows. The last rows give the October 2026 results as exact cents on the dollar.

| Measure | July 2024 (through May 2024) | October 2026 (through September 2026) |
|---|---|---|
| Colorado, interaction coefficient x 100 (no industry / industry) | 7.7 / 7.5 | 9.0 / 8.3 |
| Washington | 5.4 / 5.9 | 5.5 / 5.0 |
| California | 3.2 / 3.3 | 2.4 / 2.3 |
| Rest of the US | 2.0 / 2.0 | 1.8 / 1.8 |
| Colorado, change in cents on the dollar (no industry / industry) | not computed | 7.5 / 7.3 |
| Colorado, range across the four earnings cuts (cents) | not computed | 4.3 to 7.5 |
| Colorado dollar effect (industry / no industry) | $3,304 / $3,358 | $3,935 / $4,264 |
| Colorado men's relative change | 1.6 points (p = 0.32) | 2.5 points (p = 0.053) |
| States with a significant change at 5% (no industry / industry) | 14 / 16 | 20 / 20 |

Washington and California in 2026 use a January 2023 start date, so they are not strictly comparable with the July 2024 figures, which used January 2021 for both.

### Limitations of this update

- The earnings cut matters. The main $2,884.61 cut is fixed in nominal dollars and removes a growing share of workers; the alternative cuts in Section 6 give smaller effects, and the placebo test shows Colorado is still near the top but not always first.
- The 2025 minimum wages come from Wikipedia, and the 2026 wages from a July 2026 snapshot. No official 2025 source exists at the moment. How much this matters has not been tested yet, and a sensitivity check is planned.
- The 34 industry codes introduced by the CPS in 2025 are bridged to sectors using the nearest lower code, because IPUMS has not published a crosswalk for them.
- The October 2025 sample does not exist because the data were not collected during the federal government shutdown.
- Several states in the comparison group enacted their own pay transparency laws in 2023 to 2026, which likely understates the effect for Colorado relative to a clean control group. The extended analysis of the 14 states is planned to address this.
- Colorado has roughly 50 observations per month, so industry, urban and rural results are noisy.